# AIS Data Loading and Profiling

## Project Context

This notebook validates raw AIS (Automatic Identification System) vessel transmission data before downstream processing.

**Objective:** Determine whether the dataset is structurally reliable and suitable for cleaning and analysis.

**Dataset:** U.S. coastal AIS transmissions (January 2024)

**Scope:**
- Identifier validation (MMSI)
- Position validation (LAT, LON)
- Movement signal validation (SOG, COG, Heading)
- Vessel dimension validation (Length, Width, Draft)
- Temporal consistency checks

---

## Validation Checklist

### Identifier

[✓] MMSI format validation  
[✓] MMSI = 0 investigation  
[✓] Malformed MMSI length investigation  
[✓] MMSI metadata consistency check  

### Position

[✓] Latitude range validation (-90 to 90)  
[✓] Longitude range validation (-180 to 180)  
[✓] Coordinate continuity analysis  
[✓] Teleportation detection  

### Movement

[✓] SOG range and sentinel validation  
[✓] COG range validation  
[✓] Heading range and sentinel validation  

### Vessel Dimensions

[✓] Length range and zero-value investigation  
[✓] Width range and zero-value investigation  
[✓] Draft range and zero-value investigation  
[✓] Length-to-width ratio validation  
[✓] Draft plausibility checks  

### Temporal

[✓] Timestamp range confirmation  
[✓] Zero-time interval investigation  

---

## Reference Standards

- NOAA Marine Cadastre AIS Data Dictionary  
- U.S. Coast Guard Navigation Center — AIS Guidance  
- ITU-R M.1371 — AIS Technical Specification

In [1]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

print("Imports loaded.")

Imports loaded.


In [2]:
def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(5):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found. Run Jupyter from project root or keep /data and /notebooks in place.")

In [3]:
NOTEBOOK_PATH = Path.cwd()

PROJECT_ROOT = find_project_root(NOTEBOOK_PATH)

RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# all the AIS is for Juanuary, 2024
RUN_TS = datetime.now(timezone.utc).isoformat(timespec="seconds")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUN_TS: {RUN_TS}")
print(f"RAW_DIR exists: {RAW_DIR.exists()}")
print(f"PROCESSED_DIR exists: {PROCESSED_DIR.exists()}")

AIS_files_paths = []
for day in range(1,4):
    file_name = f"AIS_2024_01_0{day}.csv"
    file_dir = RAW_DIR / file_name
    print(f"================================================\nAIS_2024_01_0{day}.csv exists: {file_dir.exists()}")
    if file_dir.exists():
        print(f"{file_name} size (MB): {file_dir.stat().st_size/1e6:.1f}")
        AIS_files_paths.append(file_dir)

print("\nfiles Names:\n" + "\n".join(Path(p).name for p in AIS_files_paths))
print("\nPaths appended successfully")


PROJECT_ROOT: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Maritime Logistics Intelligence
RUN_TS: 2026-05-07T15:54:57+00:00
RAW_DIR exists: True
PROCESSED_DIR exists: True
AIS_2024_01_01.csv exists: True
AIS_2024_01_01.csv size (MB): 807.5
AIS_2024_01_02.csv exists: True
AIS_2024_01_02.csv size (MB): 808.2
AIS_2024_01_03.csv exists: True
AIS_2024_01_03.csv size (MB): 808.0

files Names:
AIS_2024_01_01.csv
AIS_2024_01_02.csv
AIS_2024_01_03.csv

Paths appended successfully


In [4]:
#understanding files structure, Reference file AIS_01_01_CSV
print("Reading AIS_01_01_CSV...\n")

AIS_01_01 = pd.read_csv(AIS_files_paths[0])
print("Load complete.")
print("Shape:", AIS_01_01.shape)
mem_mb = AIS_01_01.memory_usage(deep=True).sum() / 1e6
print(f"AIS_01_01 Approx memory usage: {mem_mb:.1f} MB")

Reading AIS_01_01_CSV...

Load complete.
Shape: (7296275, 17)
AIS_01_01 Approx memory usage: 1312.9 MB


In [5]:
print("Dtypes:")
print(AIS_01_01.dtypes)

print("\nObject columns:")
print(AIS_01_01.select_dtypes(include="object").columns.tolist())

print("\nTotal rows:", len(AIS_01_01))
print("\nNull counts (absolute):")
print(AIS_01_01.isna().sum())

print("\nNull percentages (%):")
null_pct = (AIS_01_01.isna().mean() * 100).round(2)
print(null_pct)


Dtypes:
MMSI                  int64
BaseDateTime            str
LAT                 float64
LON                 float64
SOG                 float64
COG                 float64
Heading             float64
VesselName              str
IMO                     str
CallSign                str
VesselType          float64
Status              float64
Length              float64
Width               float64
Draft               float64
Cargo               float64
TransceiverClass        str
dtype: object

Object columns:
['BaseDateTime', 'VesselName', 'IMO', 'CallSign', 'TransceiverClass']

Total rows: 7296275

Null counts (absolute):


C:\Users\kados\AppData\Local\Temp\ipykernel_48336\3508038655.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(AIS_01_01.select_dtypes(include="object").columns.tolist())


MMSI                      0
BaseDateTime              0
LAT                       0
LON                       0
SOG                       0
COG                       0
Heading                   0
VesselName             6956
IMO                 2245908
CallSign             827624
VesselType             3179
Status              2010774
Length                 3204
Width                  3963
Draft               2006554
Cargo               2006554
TransceiverClass          0
dtype: int64

Null percentages (%):
MMSI                 0.00
BaseDateTime         0.00
LAT                  0.00
LON                  0.00
SOG                  0.00
COG                  0.00
Heading              0.00
VesselName           0.10
IMO                 30.78
CallSign            11.34
VesselType           0.04
Status              27.56
Length               0.04
Width                0.05
Draft               27.50
Cargo               27.50
TransceiverClass     0.00
dtype: float64


In [6]:
pd.options.display.float_format = "{:,.2f}".format

In [7]:
Core_telemetry_cols = [
    "MMSI",
    "BaseDateTime",
    "LAT",
    "LON"
]
display(AIS_01_01[Core_telemetry_cols].head(20))

,MMSI,BaseDateTime,LAT,LON
0,338075892,2024-01-01T00:00:03,43.65,-70.25
1,367669550,2024-01-01T00:00:04,46.20,-123.39
2,367118980,2024-01-01T00:00:06,29.99,-90.41
3,367177840,2024-01-01T00:00:05,39.89,-75.18
4,367305420,2024-01-01T00:00:06,18.33,-64.95
5,338239081,2024-01-01T00:00:05,38.96,-76.48
6,367507960,2024-01-01T00:00:02,33.75,-118.22
7,636018568,2024-01-01T00:00:04,29.28,-94.60
8,366847780,2024-01-01T00:00:03,30.18,-87.68
9,367468580,2024-01-01T00:00:04,29.97,-90.24


In [8]:
MMSI_COL = AIS_01_01["MMSI"]

mmsi_as_str = MMSI_COL.astype(str)
mmsi_len = mmsi_as_str.str.len()

print("=== MMSI FORMAT CHECKS ===")
print("Null MMSI values:", MMSI_COL.isna().sum())
print("Negative MMSI values:", (MMSI_COL < 0).sum())
print("Zero MMSI values:", (MMSI_COL == 0).sum())
print("MMSI values shorter than 9 digits:", (mmsi_len < 9).sum())
print("MMSI values longer than 9 digits:", (mmsi_len > 9).sum())


invalid_len_mask = (mmsi_len != 9)
print("\nSample MMSI values with invalid length:")
print(MMSI_COL[invalid_len_mask].drop_duplicates().head(10).tolist())


total_rows = len(AIS_01_01)
unique_mmsi = MMSI_COL.nunique()

print("\n=== MMSI CARDINALITY ===")
print("Total rows:", total_rows)
print("Unique MMSI count:", unique_mmsi)
print("Average broadcasts per MMSI:", round(total_rows / unique_mmsi, 2) if unique_mmsi else None)


dup_broadcast_mask = AIS_01_01.duplicated(subset=["MMSI", "BaseDateTime"], keep=False)

print("\n=== MMSI + BaseDateTime DUPLICATION CHECK ===")
print("Rows involved in duplicated MMSI + BaseDateTime combinations:", dup_broadcast_mask.sum())

if dup_broadcast_mask.any():
    print("\nSample duplicated MMSI + BaseDateTime rows:")
    print(
        AIS_01_01.loc[dup_broadcast_mask, ["MMSI", "BaseDateTime", "LAT", "LON"]]
        .sort_values(["MMSI", "BaseDateTime"])
        .head(10)
    )


metadata_cols = ["VesselName", "CallSign", "IMO", "VesselType", "Length", "Width", "Draft", "TransceiverClass"]

consistency_summary = (
    AIS_01_01.groupby("MMSI")[metadata_cols]
    .nunique(dropna=True)
    .reset_index()
)

print("\n=== MMSI METADATA CONSISTENCY CHECK ===")
for col in metadata_cols:
    inconsistent_count = (consistency_summary[col] > 1).sum()
    print(f"MMSIs linked to more than one {col}: {inconsistent_count}")


suspicious_mmsi_mask = (consistency_summary[metadata_cols] > 1).any(axis=1)
suspicious_mmsis = consistency_summary.loc[suspicious_mmsi_mask, "MMSI"]

print("\nSample suspicious MMSIs with inconsistent metadata:")
print(suspicious_mmsis.head(10).tolist())

=== MMSI FORMAT CHECKS ===
Null MMSI values: 0
Negative MMSI values: 0
Zero MMSI values: 396
MMSI values shorter than 9 digits: 2479
MMSI values longer than 9 digits: 388

Sample MMSI values with invalid length:
[36968098, 36926403, 3381234, 3660489, 0, 1072211352, 4061, 99043470, 4556531, 4566545]

=== MMSI CARDINALITY ===
Total rows: 7296275
Unique MMSI count: 14868
Average broadcasts per MMSI: 490.74

=== MMSI + BaseDateTime DUPLICATION CHECK ===
Rows involved in duplicated MMSI + BaseDateTime combinations: 516

Sample duplicated MMSI + BaseDateTime rows:
              MMSI         BaseDateTime   LAT     LON
1582756  229081000  2024-01-01T06:00:00 17.69  -66.70
6600357  229081000  2024-01-01T06:00:00 17.69  -66.70
1059877  229830000  2024-01-01T03:59:59 18.35  -64.58
1273332  229830000  2024-01-01T03:59:59 18.35  -64.58
887252   232010350  2024-01-01T04:07:00 47.27 -122.44
1083310  232010350  2024-01-01T04:07:00 47.27 -122.44
2193038  232010350  2024-01-01T14:13:00 47.28 -122.44
374

### MMSI Validation — Findings

**Format anomalies**

Some MMSIs appear shorter or longer than 9 digits. Shorter values likely result from leading zeros stripped during integer parsing. MMSI = 0 and unusually long values are true invalid identifiers — investigated separately below.

**Cardinality**

7.3M rows, 14,868 unique MMSIs. Average ~491 broadcasts per vessel. Consistent with expected AIS grain.

**Duplicate timestamps**

516 rows share MMSI + BaseDateTime. Small fraction — likely repeated transmissions or timestamp precision limits.

**Metadata consistency**

Nearly all MMSIs map to stable vessel attributes. 9 MMSIs show multiple TransceiverClass values — possibly device changes. No significant identifier fragmentation detected.

---

**Outcome:** MMSI field validated. No systemic corruption. All values retained. MMSI = 0 and length anomalies flagged as metadata gaps but kept in dataset for vessel-level groupings.

---

In [9]:
print("\nSample of 0 MMSI values rows:")
display(AIS_01_01.loc[AIS_01_01["MMSI"] == 0].groupby("VesselName")[["Length","Width","CallSign"]].nunique())




Sample of 0 MMSI values rows:


,Length,Width,CallSign
VesselName,,,
CG49420,1,1,1


### MMSI = 0 Investigation

Rows with MMSI = 0 map consistently to a single vessel (VesselName = CG49420, CallSign = NWHE). This confirms a vessel broadcasting without a captured identifier rather than corrupted records.

Note: Reported dimensions (Length = 51, Width = 82) appear physically implausible — width exceeds length. Flagged for dimension validation.

---

Records retained. Tagged as "missing MMSI" for downstream filtering. These rows excluded from MMSI-based joins unless explicitly handled.

In [10]:
suspicious_dimensions_mask = AIS_01_01["Width"] > AIS_01_01["Length"]
print(f"number of rows with suspicious dimensions: {len(AIS_01_01.loc[suspicious_dimensions_mask, metadata_cols])}\n")
display(AIS_01_01.loc[suspicious_dimensions_mask, metadata_cols].head(20))

number of rows with suspicious dimensions: 37113



,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
51,LESLIE,WDE8208,IMO1218576,31.00,12.00,16.00,3.30,A
184,MISS JUANITA,WDC3039,NaN,52.00,9.00,17.00,2.20,A
904,SANTA MARIA,WDH4392,IMO9010383,60.00,41.00,47.00,0.00,A
1072,SILVER GIRL,NaN,IMO0000000,37.00,0.00,6.00,NaN,B
1701,KEN MACKENZIE,NaN,NaN,31.00,12.00,14.00,0.00,A
1746,MR JOE,WDF5575,NaN,31.00,9.00,11.00,0.00,A
1921,MISS AUDREY,WDL8837,IMO1232891,52.00,4.00,11.00,2.00,A
1935,EAGLE II,WDK8178,NaN,60.00,30.00,33.00,0.00,A
2205,MYRNA ANN,WDA7692,NaN,31.00,9.00,18.00,12.40,A
2460,RENEE T WHATLEY,WDD5243,NaN,31.00,11.00,25.00,3.00,A


Width > Length occurs in ~0.5% of rows. Localized metadata anomalies, not structural issue. No corrective action at profiling stage — flag for review if dimension analysis requires it.

In [11]:
display((AIS_01_01.loc[suspicious_dimensions_mask, "Width"] /
 AIS_01_01.loc[suspicious_dimensions_mask, "Length"]).head(20))

51     1.33
184    1.89
904    1.15
1072    inf
1701   1.17
1746   1.22
1921   2.75
1935   1.10
2205   2.00
2460   2.27
2606   2.75
2658    inf
3003   2.64
3025   1.29
3257   1.20
3331   3.00
3590   2.00
3657    inf
3757   3.12
4093   1.09
dtype: float64

Some ratios show "inf" — indicates Length = 0 cases requiring investigation.

In [12]:
zero_length_mask = AIS_01_01["Length"] == 0
print(f"number of rows with 0 length value: {len(AIS_01_01.loc[zero_length_mask, metadata_cols])}\n")
display(AIS_01_01.loc[zero_length_mask, metadata_cols].head(10))

number of rows with 0 length value: 138496



,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,PILOT BOAT SPRING PT,WDB8945,NaN,90.00,0.00,0.00,0.00,A
199,SIMCOE ISLANDER II,FISI2,NaN,99.00,0.00,0.00,0.00,A
381,QUACKERS,NaN,NaN,99.00,0.00,0.00,0.00,A
586,SAM S,WDJ2832,NaN,52.00,0.00,0.00,0.00,A
726,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,A
803,MISS EMILY,NaN,NaN,30.00,0.00,0.00,0.00,A
814,NANCE,NaN,IMO0000000,37.00,0.00,0.00,NaN,B
948,KIMMSWICK,AENA,NaN,33.00,0.00,0.00,0.00,A
978,MARLO,NaN,NaN,0.00,0.00,0.00,0.00,A
1072,SILVER GIRL,NaN,IMO0000000,37.00,0.00,6.00,NaN,B


In [13]:
zero_width_mask = AIS_01_01["Width"] == 0
print(f"number of rows with 0 width value: {len(AIS_01_01.loc[zero_width_mask, metadata_cols])}\n")
display(AIS_01_01.loc[zero_length_mask, metadata_cols].head(10))

number of rows with 0 width value: 197431



,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,PILOT BOAT SPRING PT,WDB8945,NaN,90.00,0.00,0.00,0.00,A
199,SIMCOE ISLANDER II,FISI2,NaN,99.00,0.00,0.00,0.00,A
381,QUACKERS,NaN,NaN,99.00,0.00,0.00,0.00,A
586,SAM S,WDJ2832,NaN,52.00,0.00,0.00,0.00,A
726,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,A
803,MISS EMILY,NaN,NaN,30.00,0.00,0.00,0.00,A
814,NANCE,NaN,IMO0000000,37.00,0.00,0.00,NaN,B
948,KIMMSWICK,AENA,NaN,33.00,0.00,0.00,0.00,A
978,MARLO,NaN,NaN,0.00,0.00,0.00,0.00,A
1072,SILVER GIRL,NaN,IMO0000000,37.00,0.00,6.00,NaN,B


In [14]:
zero_dimensions_mask = (AIS_01_01["Length"] == 0) & (AIS_01_01["Width"] == 0)
print(f"number of rows with 0 length and width value: {len(AIS_01_01.loc[zero_dimensions_mask, metadata_cols])}\n")
display(AIS_01_01.loc[zero_length_mask, metadata_cols].head(10))

number of rows with 0 length and width value: 134333



,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,PILOT BOAT SPRING PT,WDB8945,NaN,90.00,0.00,0.00,0.00,A
199,SIMCOE ISLANDER II,FISI2,NaN,99.00,0.00,0.00,0.00,A
381,QUACKERS,NaN,NaN,99.00,0.00,0.00,0.00,A
586,SAM S,WDJ2832,NaN,52.00,0.00,0.00,0.00,A
726,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,A
803,MISS EMILY,NaN,NaN,30.00,0.00,0.00,0.00,A
814,NANCE,NaN,IMO0000000,37.00,0.00,0.00,NaN,B
948,KIMMSWICK,AENA,NaN,33.00,0.00,0.00,0.00,A
978,MARLO,NaN,NaN,0.00,0.00,0.00,0.00,A
1072,SILVER GIRL,NaN,IMO0000000,37.00,0.00,6.00,NaN,B


### Zero Dimensions — Finding

Length = 0 and Width = 0 frequently occur together (~97% overlap). Width = 0 also occurs alone.

AIS dimensions derive from antenna offsets (bow/stern, port/starboard). Missing offsets result in zero dimensions.

Pattern confirmed as incomplete vessel metadata, not measurement errors.

---

Zero values interpreted as missing data. Rows kept in dataset but excluded from size-based filtering and ratio calculations.

---

### MMSI = 0 — Movement Analysis

In [15]:
AIS_01_01["BaseDateTime"] = pd.to_datetime(AIS_01_01["BaseDateTime"])

AIS_01_01["Date"] = AIS_01_01["BaseDateTime"].dt.date
AIS_01_01["Time"] = AIS_01_01["BaseDateTime"].dt.time
movement_cols = ["BaseDateTime","Date","Time", "LAT", "LON", "SOG", "COG", "Heading"]


In [16]:
display(
    AIS_01_01
    .loc[AIS_01_01["MMSI"] == 0, movement_cols]
    .sort_values("BaseDateTime")
    .head(20)
)

,BaseDateTime,Date,Time,LAT,LON,SOG,COG,Heading
22234,2024-01-01 00:01:59,2024-01-01,00:01:59,36.91,-76.18,0.00,171.90,171.00
42146,2024-01-01 00:04:59,2024-01-01,00:04:59,36.91,-76.18,0.00,176.70,176.00
30163,2024-01-01 00:07:59,2024-01-01,00:07:59,36.91,-76.18,0.00,181.10,181.00
68416,2024-01-01 00:13:59,2024-01-01,00:13:59,36.91,-76.18,0.00,178.10,178.00
74899,2024-01-01 00:16:59,2024-01-01,00:16:59,36.91,-76.18,0.00,179.70,179.00
460160,2024-01-01 00:22:59,2024-01-01,00:22:59,36.91,-76.18,0.00,175.70,175.00
116691,2024-01-01 00:25:59,2024-01-01,00:25:59,36.91,-76.18,0.00,176.00,176.00
136936,2024-01-01 00:31:59,2024-01-01,00:31:59,36.91,-76.18,0.00,180.50,180.00
208263,2024-01-01 00:34:59,2024-01-01,00:34:59,36.91,-76.18,0.00,179.80,179.00
627940,2024-01-01 00:37:59,2024-01-01,00:37:59,36.91,-76.18,0.00,188.60,188.00


COG and Heading values align across transmissions. Regular intervals. SOG = 0 with minimal coordinate drift indicates stationary vessel.

In [17]:
mmsi_zero_movement = (
    AIS_01_01.loc[AIS_01_01["MMSI"] == 0, ["BaseDateTime", "LAT", "LON"]]
    .sort_values("BaseDateTime")
    .copy()
)

mmsi_zero_movement["lat_diff"] = mmsi_zero_movement["LAT"].diff().abs()
mmsi_zero_movement["lon_diff"] = mmsi_zero_movement["LON"].diff().abs()

display(mmsi_zero_movement.head(30))

,BaseDateTime,LAT,LON,lat_diff,lon_diff
22234,2024-01-01 00:01:59,36.91,-76.18,NaN,NaN
42146,2024-01-01 00:04:59,36.91,-76.18,0.00,0.00
30163,2024-01-01 00:07:59,36.91,-76.18,0.00,0.00
68416,2024-01-01 00:13:59,36.91,-76.18,0.00,0.00
74899,2024-01-01 00:16:59,36.91,-76.18,0.00,0.00
460160,2024-01-01 00:22:59,36.91,-76.18,0.00,0.00
116691,2024-01-01 00:25:59,36.91,-76.18,0.00,0.00
136936,2024-01-01 00:31:59,36.91,-76.18,0.00,0.00
208263,2024-01-01 00:34:59,36.91,-76.18,0.00,0.00
627940,2024-01-01 00:37:59,36.91,-76.18,0.00,0.00


In [18]:
mmsi_zero_movement[["lat_diff", "lon_diff"]].agg(["min", "max"])

,lat_diff,lon_diff
min,0.00,0.00
max,0.00,0.00


In [19]:
AIS_01_01.loc[AIS_01_01["MMSI"] == 0, "SOG"].max()

np.float64(0.0)

In [20]:
display(
    AIS_01_01.loc[AIS_01_01["MMSI"] == 0, metadata_cols]
    .drop_duplicates()
)

,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
22234,CG49420,NWHE,NaN,51.00,82.00,12.00,1.60,B


**Conclusion:** MMSI = 0 rows show consistent metadata, minimal coordinate drift (~0.00003°), and SOG = 0. Vessel confirmed as stationary. This is a missing identifier scenario, not corrupted data.

Telemetry retained. Usable for coordinate/time analysis but excluded from vessel-keyed aggregations.

In [21]:
display(
    AIS_01_01.loc[AIS_01_01["MMSI"].astype(str).str.len() > 9]
)

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,Date,Time
33842,1072211352,2024-01-01 00:06:07,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,00:06:07
69077,1072211352,2024-01-01 00:07:39,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,00:07:39
86510,1072211352,2024-01-01 00:19:41,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,00:19:41
101583,1072211352,2024-01-01 00:22:42,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,00:22:42
113441,1072211352,2024-01-01 00:25:41,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,00:25:41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7216212,1072211352,2024-01-01 22:09:28,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,22:09:28
7240918,1072211352,2024-01-01 22:42:29,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,22:42:29
7244385,1072211352,2024-01-01 22:48:28,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,22:48:28
7266025,1072211352,2024-01-01 23:30:28,33.61,-117.93,0.00,360.00,46.00,SHAMBHALA,IMO0000000,WDG7537,37.00,NaN,30.00,6.00,NaN,NaN,B,2024-01-01,23:30:28


In [22]:
display(
    AIS_01_01.loc[AIS_01_01["MMSI"].astype(str).str.len() > 9, metadata_cols]
    .drop_duplicates()
)

,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
33842,SHAMBHALA,WDG7537,IMO0000000,37.00,30.00,6.00,NaN,B


In [23]:

mmsi_long_movement = (
    AIS_01_01.loc[AIS_01_01["MMSI"].astype(str).str.len() > 9, ["BaseDateTime", "LAT", "LON"]]
    .sort_values("BaseDateTime")
    .copy()
)

mmsi_long_movement["lat_diff"] = mmsi_long_movement["LAT"].diff().abs()
mmsi_long_movement["lon_diff"] = mmsi_long_movement["LON"].diff().abs()

display(mmsi_zero_movement.head(30))

,BaseDateTime,LAT,LON,lat_diff,lon_diff
22234,2024-01-01 00:01:59,36.91,-76.18,NaN,NaN
42146,2024-01-01 00:04:59,36.91,-76.18,0.00,0.00
30163,2024-01-01 00:07:59,36.91,-76.18,0.00,0.00
68416,2024-01-01 00:13:59,36.91,-76.18,0.00,0.00
74899,2024-01-01 00:16:59,36.91,-76.18,0.00,0.00
460160,2024-01-01 00:22:59,36.91,-76.18,0.00,0.00
116691,2024-01-01 00:25:59,36.91,-76.18,0.00,0.00
136936,2024-01-01 00:31:59,36.91,-76.18,0.00,0.00
208263,2024-01-01 00:34:59,36.91,-76.18,0.00,0.00
627940,2024-01-01 00:37:59,36.91,-76.18,0.00,0.00


In [24]:
mmsi_long_movement[["lat_diff", "lon_diff"]].agg(["min", "max"])

,lat_diff,lon_diff
min,0.00,0.00
max,0.00,0.00


In [25]:
AIS_01_01.loc[AIS_01_01["MMSI"].astype(str).str.len() > 9, "SOG"].max()

np.float64(0.4)

### Long MMSI Investigation

Records with MMSI length > 9 digits map to a single vessel:

- Vessel: SHAMBHALA
- Call Sign: WDG7537
- Transceiver: Class B
- Dimensions: 30m × 6m

Movement analysis confirms minimal coordinate changes (max ~0.00005°) and SOG near zero (max ~0.4 kn). Vessel is stationary.

---

Anomalous MMSI length is a formatting issue, not corrupted telemetry. Data preserved. MMSI normalization optional during curation. Single vessel — no effect on identifier joins.

In [26]:
short_mmsi_mask = (
    (AIS_01_01["MMSI"].astype(str).str.len() < 9) &
    (AIS_01_01["MMSI"] != 0)
)
display(
    AIS_01_01.loc[short_mmsi_mask].head(10)
)

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,Date,Time
726,36968098,2024-01-01 00:00:07,36.95,-76.33,0.10,128.80,511.00,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,A,2024-01-01,00:00:07
2942,36926403,2024-01-01 00:00:18,30.39,-81.41,0.00,104.20,285.00,WARSHIP 25,NaN,NMNE,35.00,5.00,0.00,0.00,0.00,35.00,A,2024-01-01,00:00:18
4129,36968098,2024-01-01 00:01:17,36.95,-76.33,0.10,123.60,511.00,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,A,2024-01-01,00:01:17
4919,3381234,2024-01-01 00:01:03,34.72,-76.71,0.60,98.50,511.00,ZEEPAARD,IMO0000000,BO12345,36.00,NaN,0.00,0.00,NaN,NaN,B,2024-01-01,00:01:03
13360,3381234,2024-01-01 00:03:03,34.72,-76.71,0.60,98.50,511.00,ZEEPAARD,IMO0000000,BO12345,36.00,NaN,0.00,0.00,NaN,NaN,B,2024-01-01,00:03:03
14614,36968098,2024-01-01 00:02:27,36.95,-76.33,0.00,126.30,511.00,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,A,2024-01-01,00:02:27
15263,3660489,2024-01-01 00:01:47,27.37,-89.92,0.00,311.00,0.00,NEPTUNE TLP,IMO0745081,WQGV318,99.00,5.00,89.00,60.00,25.50,99.00,A,2024-01-01,00:01:47
16682,36926403,2024-01-01 00:03:19,30.39,-81.41,0.00,94.30,285.00,WARSHIP 25,NaN,NMNE,35.00,5.00,0.00,0.00,0.00,35.00,A,2024-01-01,00:03:19
21737,3381234,2024-01-01 00:05:03,34.72,-76.71,0.60,0.00,511.00,ZEEPAARD,IMO0000000,BO12345,36.00,NaN,0.00,0.00,NaN,NaN,B,2024-01-01,00:05:03
23991,36968098,2024-01-01 00:04:48,36.95,-76.33,0.10,125.60,511.00,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,A,2024-01-01,00:04:48


In [27]:
display(
    AIS_01_01.loc[short_mmsi_mask, ["MMSI"] + metadata_cols]
    .drop_duplicates()
)

,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
726,36968098,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,A
2942,36926403,WARSHIP 25,NMNE,NaN,35.00,0.00,0.00,0.00,A
4919,3381234,ZEEPAARD,BO12345,IMO0000000,36.00,0.00,0.00,NaN,B
15263,3660489,NEPTUNE TLP,WQGV318,IMO0745081,99.00,89.00,60.00,25.50,A
36129,4061,BOOSTER 9000,0000000,IMO0000000,33.00,0.00,0.00,NaN,B
1268886,99043470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A
5657738,4556531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
5824125,4566545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
6094904,4566548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B


### Short MMSI Investigation

Records with MMSI length under 9 digits map to distinct vessels with stable metadata and localized movement patterns. Short lengths result from leading zeros stripped during integer parsing rather than corrupted identifiers.

---

Parsing artifact, not data error. No reformatting needed. Vessel identification unaffected.

In [28]:
max_sog_per_mmsi = (
    AIS_01_01.loc[short_mmsi_mask]
    .groupby("MMSI")["SOG"]
    .max()
    .sort_values(ascending=False)
)

display(max_sog_per_mmsi)

MMSI
4566545    10.50
4566548     3.40
4556531     2.30
3381234     1.20
36968098    0.20
4061        0.10
3660489     0.10
36926403    0.00
99043470    0.00
Name: SOG, dtype: float64

In [29]:
selected_mmsi = 4566545   
vessel_df = (
    AIS_01_01[AIS_01_01["MMSI"] == selected_mmsi]
    .sort_values("BaseDateTime")
    [["BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading"]]
    .copy()
)

print(f"MMSI: {selected_mmsi}")
print(f"Number of rows: {len(vessel_df)}")

display(vessel_df.head(20))
display(vessel_df.tail(20))

MMSI: 4566545
Number of rows: 9


,BaseDateTime,LAT,LON,SOG,COG,Heading
6420925,2024-01-01 21:45:27,35.57,-74.69,10.50,41.30,87.00
5824125,2024-01-01 21:53:27,35.58,-74.68,3.90,52.30,87.00
7224492,2024-01-01 22:17:27,35.59,-74.66,3.50,74.60,87.00
6015850,2024-01-01 22:25:28,35.59,-74.66,2.50,62.60,87.00
6005476,2024-01-01 22:33:28,35.60,-74.65,3.20,73.40,87.00
6037331,2024-01-01 22:37:28,35.60,-74.65,3.10,63.00,87.00
6276207,2024-01-01 23:33:29,35.63,-74.60,3.10,37.50,87.00
6285908,2024-01-01 23:35:29,35.63,-74.60,3.10,37.50,87.00
6303942,2024-01-01 23:37:29,35.63,-74.60,3.70,47.90,87.00


,BaseDateTime,LAT,LON,SOG,COG,Heading
6420925,2024-01-01 21:45:27,35.57,-74.69,10.50,41.30,87.00
5824125,2024-01-01 21:53:27,35.58,-74.68,3.90,52.30,87.00
7224492,2024-01-01 22:17:27,35.59,-74.66,3.50,74.60,87.00
6015850,2024-01-01 22:25:28,35.59,-74.66,2.50,62.60,87.00
6005476,2024-01-01 22:33:28,35.60,-74.65,3.20,73.40,87.00
6037331,2024-01-01 22:37:28,35.60,-74.65,3.10,63.00,87.00
6276207,2024-01-01 23:33:29,35.63,-74.60,3.10,37.50,87.00
6285908,2024-01-01 23:35:29,35.63,-74.60,3.10,37.50,87.00
6303942,2024-01-01 23:37:29,35.63,-74.60,3.70,47.90,87.00


In [30]:
vessel_df["lat_diff"] = vessel_df["LAT"].diff().abs()
vessel_df["lon_diff"] = vessel_df["LON"].diff().abs()

vessel_df["coord_step_change"] = (
    vessel_df["lat_diff"] +
    vessel_df["lon_diff"]
)

display(vessel_df.head(20))

,BaseDateTime,LAT,LON,SOG,COG,Heading,lat_diff,lon_diff,coord_step_change
6420925,2024-01-01 21:45:27,35.57,-74.69,10.50,41.30,87.00,NaN,NaN,NaN
5824125,2024-01-01 21:53:27,35.58,-74.68,3.90,52.30,87.00,0.01,0.01,0.02
7224492,2024-01-01 22:17:27,35.59,-74.66,3.50,74.60,87.00,0.01,0.02,0.03
6015850,2024-01-01 22:25:28,35.59,-74.66,2.50,62.60,87.00,0.00,0.01,0.01
6005476,2024-01-01 22:33:28,35.60,-74.65,3.20,73.40,87.00,0.00,0.01,0.01
6037331,2024-01-01 22:37:28,35.60,-74.65,3.10,63.00,87.00,0.00,0.00,0.01
6276207,2024-01-01 23:33:29,35.63,-74.60,3.10,37.50,87.00,0.03,0.04,0.07
6285908,2024-01-01 23:35:29,35.63,-74.60,3.10,37.50,87.00,0.00,0.00,0.00
6303942,2024-01-01 23:37:29,35.63,-74.60,3.70,47.90,87.00,0.00,0.00,0.01


In [31]:
total_coord_movement = vessel_df["coord_step_change"].sum()

print("Total coordinate movement:", total_coord_movement)

Total coordinate movement: 0.1534599999999955


In [32]:
start_lat = vessel_df["LAT"].iloc[0]
start_lon = vessel_df["LON"].iloc[0]

end_lat = vessel_df["LAT"].iloc[-1]
end_lon = vessel_df["LON"].iloc[-1]

start_end_displacement = (
    abs(end_lat - start_lat) +
    abs(end_lon - start_lon)
)

print("Start-end displacement:", start_end_displacement)

Start-end displacement: 0.1534599999999955


### Malformed MMSI — Conclusion

All malformed MMSI cases (zero, short, long) correspond to stationary or locally operating vessels. Identifier formatting issues affecting metadata, not telemetry corruption.

All anomalies explained. Dataset identifier quality confirmed. Full dataset retained — edge cases documented for downstream reference.

---

## Vessel Dimension Validation

Zero-value dimensions were investigated above. Now validating overall dimension ranges and structural plausibility.

In [33]:
dimension_cols = ["Length", "Width", "Draft"]

for col in dimension_cols:
    print(f"\n{col} range:")
    print("Min:", AIS_01_01[col].min())
    print("Max:", AIS_01_01[col].max())


Length range:
Min: 0.0
Max: 512.0

Width range:
Min: 0.0
Max: 100.0

Draft range:
Min: 0.0
Max: 25.5


In [34]:
small_dimension_mask = (
    (AIS_01_01["Length"] > 0) &
    (AIS_01_01["Width"] > 0) &
    (
        (AIS_01_01["Length"] < 10) |
        (AIS_01_01["Width"] < 3)
    )
)

small_dimension_metadata = (
    AIS_01_01.loc[
            small_dimension_mask,
            ["MMSI"]+metadata_cols
        ]
    
    .drop_duplicates()
    .sort_values(["Length", "Width", "VesselName"])
)

print(f"Number of affected vessels: {len(small_dimension_metadata)}\n")

display(small_dimension_metadata.head(30))

Number of affected vessels: 661



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
10741,338229689,KNOT FOUR REEL,NaN,IMO0000000,37.00,1.00,1.00,NaN,B
49267,316035714,GRACIE,GRACIE,IMO0000000,37.00,1.00,5.00,NaN,B
9629,338152894,OPTIONAL NECESSITY,NaN,IMO0000000,37.00,1.00,5.00,NaN,B
545354,368028770,AB 8,WDI9723,IMO0000000,90.00,2.00,2.00,NaN,B
284973,109070092,SNURREVAD,1,IMO0000000,30.00,2.00,2.00,NaN,B
426,338127859,VOBAL,NaN,IMO0000000,20.00,2.00,2.00,NaN,B
3964,338395345,LEXANY IV,NaN,IMO0000000,37.00,3.00,1.00,NaN,B
5418443,338402353,21 WHALER,NaN,IMO0000000,37.00,3.00,2.00,NaN,B
9052,316014670,FARANDA,NaN,IMO0000000,37.00,3.00,4.00,NaN,B
6885,316051686,NOUS 2,NaN,IMO0000000,36.00,4.00,1.00,NaN,B


In [35]:
small_dim_type_36_37_mask = (
    small_dimension_mask &
    AIS_01_01["VesselType"].isin([36, 37])
)

small_dim_type_36_37_metadata = (
    AIS_01_01.loc[
        small_dim_type_36_37_mask,
        ["MMSI"]+metadata_cols
    ]
    .drop_duplicates()
    .sort_values(["VesselType", "Length", "Width"])
)

print(f"Number of affected vessels (Type 36 & 37): {len(small_dim_type_36_37_metadata)}\n")

display(small_dim_type_36_37_metadata.head(30))

Number of affected vessels (Type 36 & 37): 433



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
6885,316051686,NOUS 2,NaN,IMO0000000,36.00,4.00,1.00,NaN,B
17382,367780630,QUINN,WDJ4737,IMO0000000,36.00,4.00,4.00,NaN,B
4008,367026650,SHEET MUSIC,WDC4561,IMO0000000,36.00,5.00,1.00,NaN,B
2950,368048610,LOKI,WDK3775,IMO0000000,36.00,6.00,2.00,NaN,B
109178,368219350,DESTINY,WDM5799,IMO0000000,36.00,6.00,4.00,NaN,B
4044,338461508,UGLY DUCKLING,NaN,IMO0000000,36.00,7.00,2.00,NaN,B
5715,338391954,TEMPEST,NaN,IMO0000000,36.00,7.00,2.00,NaN,B
13599,367664070,EVENINGSTAR,WDH9037,IMO0000000,36.00,7.00,2.00,NaN,B
345991,338356273,GRACEFUL GHOST,WCV3022,IMO0000000,36.00,7.00,2.00,NaN,B
6581,338482434,SARANADE,NaN,IMO0000000,36.00,7.00,4.00,NaN,B


In [36]:
small_dim_other_types_mask = (
    small_dimension_mask &
    ~AIS_01_01["VesselType"].isin([36, 37])
)

small_dim_other_types_metadata = (
    AIS_01_01.loc[
        small_dim_other_types_mask,
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates()
    .sort_values(["VesselType", "Length", "Width"])
)

print(f"Number of affected vessels (other types): {len(small_dim_other_types_metadata)}\n")

display(small_dim_other_types_metadata.head(30))

Number of affected vessels (other types): 228



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
3622138,368181960,LEYTE GULF,WDL9756,IMO5118723,0.00,4.00,2.00,0.00,A
4118398,338926661,NaN,NaN,NaN,0.00,8.00,3.00,0.00,A
3827468,369990451,CG29451,NaN,NaN,0.00,8.00,4.00,1.00,A
162510,338473618,CHALLENGER II,NaN,NaN,0.00,9.00,4.00,0.00,A
4193888,338439617,EMILY,NaN,NaN,0.00,9.00,6.00,0.00,A
426,338127859,VOBAL,NaN,IMO0000000,20.00,2.00,2.00,NaN,B
284973,109070092,SNURREVAD,1,IMO0000000,30.00,2.00,2.00,NaN,B
3962711,367396530,MY LEE,WDE7700,IMO0000000,30.00,5.00,4.00,NaN,B
7675,338358224,BEL SOGNO,NaN,IMO0000000,30.00,7.00,2.00,NaN,B
6547,338486227,WEST BOUND,NaN,IMO0000000,30.00,8.00,2.00,NaN,B


In [37]:
print("VesselType range within small-dimension cases:")
print("Min:", AIS_01_01.loc[small_dimension_mask, "VesselType"].min())
print("Max:", AIS_01_01.loc[small_dimension_mask, "VesselType"].max(), "\n")


type_30_40_mask = (
    small_dimension_mask &
    AIS_01_01["VesselType"].between(30, 40)
)

number_of_vessels_30_40 = AIS_01_01.loc[
    type_30_40_mask,
    "MMSI"
].nunique()

print(f"Number of affected vessels with VesselType between 30 and 40 (inclusive): {number_of_vessels_30_40}")

number_of_vessels_other_types = AIS_01_01.loc[
    small_dimension_mask & ~AIS_01_01["VesselType"].between(30, 40),
    "MMSI"
].nunique()

print(f"Number of affected vessels with VesselType outside 30–40: {number_of_vessels_other_types}")


AIS_01_01.loc[
    type_30_40_mask,
    "VesselType"
].value_counts().sort_index()

VesselType range within small-dimension cases:
Min: 0.0
Max: 255.0 

Number of affected vessels with VesselType between 30 and 40 (inclusive): 487
Number of affected vessels with VesselType outside 30–40: 172


VesselType
30.00     6522
31.00    11157
32.00      631
33.00     1100
36.00    25295
37.00    74633
38.00       96
40.00      809
Name: count, dtype: int64

Small-dimension cases appear frequently in VesselType 30–40. This is expected — VesselType 30–40 is one of the most common groups in the overall dataset. Focusing validation on this group first.

In [38]:
type_30_40_metadata = (
    AIS_01_01.loc[
        type_30_40_mask,
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates()
    .sort_values(["VesselType", "Length", "Width", "VesselName"])
)

print(f"Number of affected vessels in VesselType 30–40 group: {len(type_30_40_metadata)}\n")

display(type_30_40_metadata.head(30))

Number of affected vessels in VesselType 30–40 group: 487



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
284973,109070092,SNURREVAD,1,IMO0000000,30.00,2.00,2.00,NaN,B
3962711,367396530,MY LEE,WDE7700,IMO0000000,30.00,5.00,4.00,NaN,B
7675,338358224,BEL SOGNO,NaN,IMO0000000,30.00,7.00,2.00,NaN,B
6547,338486227,WEST BOUND,NaN,IMO0000000,30.00,8.00,2.00,NaN,B
4572398,338429919,GREY WOLF,WDC9067,IMO0000000,30.00,8.00,3.00,NaN,B
4235974,338229216,SALT SHAKER II,NaN,IMO0000000,30.00,8.00,3.00,NaN,B
2855610,367482110,SOUTHERN CROSS,WDM4038,IMO8687971,30.00,8.00,3.00,NaN,B
18104,338187822,COUNTING COUP,NaN,IMO0000000,30.00,8.00,4.00,NaN,B
5089791,367118570,GOLDEN GATE,WDD2688,IMO0000000,30.00,8.00,4.00,NaN,B
97412,338204529,MAD PROPS,NaN,IMO0000000,30.00,8.00,4.00,NaN,B


In [39]:
type_30_40_ratio_df = (
    AIS_01_01.loc[
        type_30_40_mask &
        (AIS_01_01["Length"] > 0) &
        (AIS_01_01["Width"] > 0),
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates()
    .copy()
)

# add ratio column so it is visible in the dataset
type_30_40_ratio_df["length_width_ratio"] = (
    type_30_40_ratio_df["Length"] /
    type_30_40_ratio_df["Width"]
)


suspicious_ratio_df = (
    type_30_40_ratio_df[
        (type_30_40_ratio_df["length_width_ratio"] > 10) |
        (type_30_40_ratio_df["length_width_ratio"] < 1)
    ]
    .sort_values(
        ["length_width_ratio", "Length", "Width"],
        ascending=[False, True, True]
    )
)

normal_ratio_count = len(type_30_40_ratio_df) - len(suspicious_ratio_df)

print(
    f"Number of affected vessels with suspicious Length/Width ratio: "
    f"{len(suspicious_ratio_df)}"
)

print(
    f"Number of affected vessels left as structurally plausible for now: "
    f"{normal_ratio_count}\n"
)

display(suspicious_ratio_df.head(30))

Number of affected vessels with suspicious Length/Width ratio: 22
Number of affected vessels left as structurally plausible for now: 465



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
16027,367732790,ENCORE V,WDI7847,IMO0000000,37.00,24.00,1.00,NaN,B,24.00
4709936,338244502,NUTTHOUSE,NaN,IMO0000000,37.00,17.00,1.00,NaN,B,17.00
12500,367465730,LADY J,WDF5651,IMO0000000,37.00,16.00,1.00,NaN,B,16.00
1617343,338446637,ARCH ANGEL,NaN,IMO0000000,36.00,13.00,1.00,NaN,B,13.00
12910,316011765,DUTCH WANDERER,CFG7589,IMO0000000,36.00,11.00,1.00,NaN,B,11.00
1811879,368055670,BAREFOOT,WDK4485,IMO0000000,36.00,11.00,1.00,NaN,B,11.00
4261086,368341050,BELLADRUM,WDP2967,IMO0000000,36.00,11.00,1.00,NaN,B,11.00
8279,338108662,CASTIGO,CUWM8,IMO0000000,37.00,5.00,6.00,NaN,B,0.83
1746,367464840,MR JOE,WDF5575,NaN,31.00,9.00,11.00,0.00,A,0.82
84061,338234391,SPRAY,NaN,IMO0000000,37.00,8.00,10.00,NaN,B,0.80


### Length-to-Width Ratio — Interpretation

**Thresholds used:**
- Ratio 2–10: structurally plausible for most vessels
- Ratio < 1 or > 10: flagged for review

22 vessels in the small-dimension group (VesselType 30–40) show unusual ratios. No consistent pattern beyond the ratio itself.

---

Flagged vessels need context-specific review, not automatic exclusion. All records kept. Ratio flags passed to downstream analysis for optional filtering.

In [40]:
other_types_ratio_df = (
    AIS_01_01.loc[
        small_dimension_mask &
        ~AIS_01_01["VesselType"].between(30, 40) &
        (AIS_01_01["Length"] > 0) &
        (AIS_01_01["Width"] > 0),
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates()
    .copy()
)

other_types_ratio_df["length_width_ratio"] = (
    other_types_ratio_df["Length"] /
    other_types_ratio_df["Width"]
)

suspicious_other_ratio_df = (
    other_types_ratio_df[
        (other_types_ratio_df["length_width_ratio"] < 1) |
        (other_types_ratio_df["length_width_ratio"] > 10)
    ]
    .sort_values(
        ["length_width_ratio", "Length", "Width"],
        ascending=[False, True, True]
    )
)

normal_other_ratio_count = len(other_types_ratio_df) - len(suspicious_other_ratio_df)

print(
    f"Number of affected vessels outside VesselType 30–40 with suspicious Length/Width ratio: "
    f"{len(suspicious_other_ratio_df)}"
)

print(
    f"Number of affected vessels outside VesselType 30–40 left as structurally plausible for now: "
    f"{normal_other_ratio_count}\n"
)

display(suspicious_other_ratio_df)

Number of affected vessels outside VesselType 30–40 with suspicious Length/Width ratio: 10
Number of affected vessels outside VesselType 30–40 left as structurally plausible for now: 164



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
21745,368021320,CAPT CURTIS,WDJ8942,IMO0000000,52.00,7.00,8.00,NaN,B,0.88
1411655,368248090,BUBBA E,WDM8953,IMO0000000,52.00,7.00,8.00,NaN,B,0.88
184,367001890,MISS JUANITA,WDC3039,NaN,52.00,9.00,17.00,2.20,A,0.53
3590,338473358,4 OF HARTS,NaN,IMO0000000,90.00,7.00,14.00,NaN,B,0.50
27812,367199280,M/V REED,WCX7530,IMO0000000,52.00,7.00,16.00,NaN,B,0.44
1921,368173380,MISS AUDREY,WDL8837,IMO1232891,52.00,4.00,11.00,2.00,A,0.36
2606,316040584,GRAPPLE,VY9372,IMO0000000,52.00,4.00,11.00,NaN,B,0.36
5995,316044191,JUDY J,WY5355,NaN,52.00,7.00,20.00,0.00,A,0.35
3757,368271360,SAN MARCOS,WDN3405,IMO5107669,79.00,8.00,25.00,0.00,A,0.32
1376183,368271360,SAN MARCOS,WDN3405,IMO5107669,79.00,8.00,25.00,2.00,A,0.32


In [41]:
combined_suspicious_ratio_df = (
    pd.concat(
        [
            suspicious_ratio_df,
            suspicious_other_ratio_df
        ],
        ignore_index=True
    )
    .drop_duplicates(subset=["MMSI"])
    .sort_values(
        ["length_width_ratio", "Length", "Width"],
        ascending=[False, True, True]
    )
)

print(
    f"Total number of vessels flagged with suspicious Length/Width ratio: "
    f"{len(combined_suspicious_ratio_df)}\n"
)

display(combined_suspicious_ratio_df)

Total number of vessels flagged with suspicious Length/Width ratio: 31



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
0,367732790,ENCORE V,WDI7847,IMO0000000,37.00,24.00,1.00,NaN,B,24.00
1,338244502,NUTTHOUSE,NaN,IMO0000000,37.00,17.00,1.00,NaN,B,17.00
2,367465730,LADY J,WDF5651,IMO0000000,37.00,16.00,1.00,NaN,B,16.00
3,338446637,ARCH ANGEL,NaN,IMO0000000,36.00,13.00,1.00,NaN,B,13.00
4,316011765,DUTCH WANDERER,CFG7589,IMO0000000,36.00,11.00,1.00,NaN,B,11.00
5,368055670,BAREFOOT,WDK4485,IMO0000000,36.00,11.00,1.00,NaN,B,11.00
6,368341050,BELLADRUM,WDP2967,IMO0000000,36.00,11.00,1.00,NaN,B,11.00
22,368021320,CAPT CURTIS,WDJ8942,IMO0000000,52.00,7.00,8.00,NaN,B,0.88
23,368248090,BUBBA E,WDM8953,IMO0000000,52.00,7.00,8.00,NaN,B,0.88
7,338108662,CASTIGO,CUWM8,IMO0000000,37.00,5.00,6.00,NaN,B,0.83


Ratio checks complete for small-dimension cases. No consistent anomaly pattern. Moving to larger vessel category.

In [42]:
large_dimension_mask = ~small_dimension_mask

print(
    f"Number of vessels outside small-dimension cases: "
    f"{AIS_01_01.loc[large_dimension_mask, 'MMSI'].nunique()}"
)

display(
    AIS_01_01.loc[
        large_dimension_mask,
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates()
    .head(30)
)

Number of vessels outside small-dimension cases: 14209


,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,338075892,PILOT BOAT SPRING PT,WDB8945,NaN,90.00,0.00,0.00,0.00,A
1,367669550,ALASKA CHALLENGER,WDH9586,IMO7938024,30.00,30.00,8.00,0.00,A
2,367118980,CAPT J A MORGAN,WDD2725,IMO1186680,31.00,115.00,34.00,3.00,A
3,367177840,BART TURECAMO,WBR4464,IMO7338808,52.00,33.00,5.00,0.00,A
4,367305420,DOROTHY MORAN,WXU4654,IMO7716995,52.00,33.00,11.00,0.00,A
5,338239081,JAHAZI,NaN,IMO0000000,36.00,12.00,4.00,NaN,B
6,367507960,DB SALTA VERDE,WDF9705,NaN,90.00,56.00,15.00,2.00,A
7,636018568,VIVIT ALTAIS,D5QH3,IMO9840879,80.00,230.00,32.00,11.80,A
8,366847780,PACIFIC DAWN,WDA7844,IMO7400467,31.00,30.00,8.00,5.00,A
9,367468580,EMMA CLAIRE,WDF5910,NaN,31.00,20.00,7.00,2.00,A


In [43]:
print("VesselType range within larger-dimension cases:")

print(
    "Min:",
    AIS_01_01.loc[
        large_dimension_mask,
        "VesselType"
    ].min()
)

print(
    "Max:",
    AIS_01_01.loc[
        large_dimension_mask,
        "VesselType"
    ].max(),
    "\n"
)

VesselType range within larger-dimension cases:
Min: 0.0
Max: 99.0 



In [44]:
large_ratio_df = (
    AIS_01_01.loc[
        large_dimension_mask &
        (AIS_01_01["Length"] > 0) &
        (AIS_01_01["Width"] > 0),
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates()
    .copy()
)

large_ratio_df["length_width_ratio"] = (
    large_ratio_df["Length"] /
    large_ratio_df["Width"]
)

suspicious_large_ratio_df = (
    large_ratio_df[
        (large_ratio_df["length_width_ratio"] < 1) |
        (large_ratio_df["length_width_ratio"] > 10)
    ]
    .sort_values(
        ["length_width_ratio", "Length", "Width"],
        ascending=[False, True, True]
    )
)

normal_large_ratio_count = (
    len(large_ratio_df) -
    len(suspicious_large_ratio_df)
)

print(
    f"Number of vessels with suspicious Length/Width ratio (larger-dimension cases): "
    f"{len(suspicious_large_ratio_df)}"
)

print(
    f"Number of vessels left as structurally plausible for now: "
    f"{normal_large_ratio_count}\n"
)

display(suspicious_large_ratio_df.head(30))

Number of vessels with suspicious Length/Width ratio (larger-dimension cases): 276
Number of vessels left as structurally plausible for now: 14270



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
273564,338349131,HAZEL LILLIE,HAZEL,IMO0000000,36.00,342.00,3.00,NaN,B,114.00
2288,367690950,RICHARD B.,WDI3694,NaN,31.00,390.00,6.00,0.00,A,65.00
7217,368090950,JUBILEE,WDK8115,IMO0000000,37.00,184.00,5.00,NaN,B,36.80
27874,316022594,SPIRIT WIND 1,NaN,IMO0000000,37.00,106.00,3.00,NaN,B,35.33
2623804,367389000,M/Y K,WDI5807,IMO0000000,37.00,203.00,6.00,NaN,B,33.83
18084,368330870,CALM CHAOS,NaN,IMO0000000,37.00,170.00,6.00,NaN,B,28.33
1343,368059490,BAILEY,WDK4879,NaN,52.00,302.00,11.00,2.90,A,27.45
121010,368296000,HAWAII,WDN7088,IMO1245560,31.00,276.00,11.00,6.10,A,25.09
28910,316004971,EMPRESS III,WCU6408,IMO0000000,60.00,88.00,4.00,NaN,B,22.00
15157,369596000,21 SEA SANDS,WDJ2193,IMO0000000,37.00,144.00,7.00,NaN,B,20.57


In [45]:
print("VesselType distribution within suspicious larger-dimension cases:\n")

AIS_01_01.loc[
    suspicious_large_ratio_df.index,
    "VesselType"
].value_counts().sort_index()

VesselType distribution within suspicious larger-dimension cases:



VesselType
0.00       1
30.00     11
31.00    146
34.00      1
36.00      9
37.00     24
51.00      1
52.00     26
53.00      1
56.00      1
57.00     25
60.00      6
70.00     13
79.00      5
90.00      5
99.00      1
Name: count, dtype: int64

VesselType 30–40 appears in unusual ratio cases. This reflects baseline prevalence — these types are common in the dataset overall. Isolating for review.

In [46]:
type_30_40_large_suspicious_df = (
    suspicious_large_ratio_df[
        suspicious_large_ratio_df["VesselType"].between(30, 40)
    ]
    .sort_values(
        ["VesselType", "length_width_ratio", "Length", "Width"]
    )
)

print(
    f"Number of suspicious vessels within VesselType 30–40 "
    f"(larger-dimension cases): "
    f"{len(type_30_40_large_suspicious_df)}\n"
)

display(type_30_40_large_suspicious_df.head(30))

Number of suspicious vessels within VesselType 30–40 (larger-dimension cases): 191



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
20342,368026770,VAK 1,WDJ9511,IMO0000000,30.00,68.00,6.00,NaN,B,11.33
1319,367538940,RITA ANN,WDG4670,IMO101240460,30.00,205.00,18.00,3.10,A,11.39
977,367539060,SUSAN ELIZABETH,WDG4681,IMO1030000,30.00,206.00,18.00,2.60,A,11.44
1742,367395010,SALVATION,WDE7579,IMO101218049,30.00,207.00,18.00,3.00,A,11.50
2450,367514360,SAN PEDRO,WDG2312,NaN,30.00,211.00,18.00,3.10,A,11.72
7251,367614790,LUCKY TJ,WDH4162,IMO7642883,30.00,71.00,6.00,NaN,B,11.83
3143505,367059330,REDEEMER,WDC6664,IMO101174264,30.00,266.00,22.00,2.90,A,12.09
2166,368047580,JONES,WDK3678,NaN,30.00,206.00,17.00,2.30,A,12.12
1463536,368135990,CAPT SCOTT,WDL4904,IMO8938681,30.00,97.00,8.00,0.00,A,12.12
6025,368305410,KAHI,WDN7111,IMO0000000,30.00,111.00,9.00,NaN,B,12.33


Some flagged vessels (ratio 10–15) are operationally plausible — long narrow passenger/service vessels. Treated as review cases, not errors. No exclusion. Documented for domain-specific validation if needed.

In [47]:
large_outside_30_40_suspicious_df = (
    suspicious_large_ratio_df[
        ~suspicious_large_ratio_df["VesselType"].between(30, 40)
    ]
    .sort_values(["VesselType", "length_width_ratio", "Length", "Width"])
)

print(
    f"Number of suspicious vessels outside VesselType 30–40 "
    f"(larger-dimension cases): "
    f"{len(large_outside_30_40_suspicious_df)}\n"
)

display(large_outside_30_40_suspicious_df.head(30))

Number of suspicious vessels outside VesselType 30–40 (larger-dimension cases): 85



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
47704,367143520,LOOP GI59,WRW2275,IMO1020304,0.00,101.00,8.00,0.00,A,12.62
11571,366999523,CG KANKAKEE,NAMR,NaN,51.00,63.00,6.00,1.40,A,10.50
3003,366995760,BAILEY MATTHEW,WDC2628,IMO6714172,52.00,25.00,66.00,3.00,A,0.38
30344,366951660,GLADYS M,WYR9477,NaN,52.00,12.00,14.00,11.00,A,0.86
211411,316009423,RADIUM YELLOWKNIFE,CFN4606,IMO5288956,52.00,36.00,40.00,1.30,A,0.90
4093,367309430,ARCTIC DAWN,WDN5235,IMO7404047,52.00,22.00,24.00,0.00,A,0.92
1984,367480250,OLIVE L MOORE,WDF7019,IMO8635227,52.00,222.00,22.00,6.40,A,10.09
13125,368145870,LILLIANNE D,WDL5939,NaN,52.00,123.00,12.00,3.10,A,10.25
5456,367487570,BRADSHAW MCKEE,WDF7733,IMO7644312,52.00,164.00,16.00,6.40,A,10.25
7486,368247000,INTEGRITY,WDD7905,IMO9369394,52.00,208.00,20.00,6.70,A,10.40


In [48]:
type_0_suspicious_df = (
    suspicious_large_ratio_df[
        suspicious_large_ratio_df["VesselType"] == 0
    ]
    .sort_values(["Length", "Width"])
)

print(
    f"Number of suspicious vessels with VesselType = 0 "
    f"(larger-dimension cases): "
    f"{len(type_0_suspicious_df)}\n"
)

display(type_0_suspicious_df)

Number of suspicious vessels with VesselType = 0 (larger-dimension cases): 1



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
47704,367143520,LOOP GI59,WRW2275,IMO1020304,0.00,101.00,8.00,0.00,A,12.62


In [49]:
all_dimension_outliers_df = (
    pd.concat(
        [
            suspicious_ratio_df,          # small-dimension cases
            suspicious_large_ratio_df     # larger-dimension cases
        ]
    )
    .drop_duplicates(subset="MMSI")
    .sort_values(["VesselType", "length_width_ratio", "Length", "Width"])
)

print(
    f"Total number of vessels flagged by Length/Width validation: "
    f"{len(all_dimension_outliers_df)}\n"
)

display(all_dimension_outliers_df.head(30))

Total number of vessels flagged by Length/Width validation: 284



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass,length_width_ratio
47704,367143520,LOOP GI59,WRW2275,IMO1020304,0.00,101.00,8.00,0.00,A,12.62
20342,368026770,VAK 1,WDJ9511,IMO0000000,30.00,68.00,6.00,NaN,B,11.33
1319,367538940,RITA ANN,WDG4670,IMO101240460,30.00,205.00,18.00,3.10,A,11.39
977,367539060,SUSAN ELIZABETH,WDG4681,IMO1030000,30.00,206.00,18.00,2.60,A,11.44
1742,367395010,SALVATION,WDE7579,IMO101218049,30.00,207.00,18.00,3.00,A,11.50
2450,367514360,SAN PEDRO,WDG2312,NaN,30.00,211.00,18.00,3.10,A,11.72
7251,367614790,LUCKY TJ,WDH4162,IMO7642883,30.00,71.00,6.00,NaN,B,11.83
3143505,367059330,REDEEMER,WDC6664,IMO101174264,30.00,266.00,22.00,2.90,A,12.09
2166,368047580,JONES,WDK3678,NaN,30.00,206.00,17.00,2.30,A,12.12
1463536,368135990,CAPT SCOTT,WDL4904,IMO8938681,30.00,97.00,8.00,0.00,A,12.12


In [50]:
print("VesselType distribution within all dimension outliers:\n")

all_dimension_outliers_df["VesselType"] \
    .value_counts() \
    .sort_index()

VesselType distribution within all dimension outliers:



VesselType
0.00       1
30.00     11
31.00    141
32.00      1
34.00      1
36.00     13
37.00     39
51.00      1
52.00     26
53.00      1
56.00      1
57.00     24
60.00      6
70.00     10
79.00      2
90.00      5
99.00      1
Name: count, dtype: int64

### Length and Width Validation — Summary

Length-to-width ratios flagged a small subset of vessels for review. VesselType 30–40 appears often in this subset because it is one of the most common vessel groups in the dataset — its prevalence here is proportional to its baseline share, not a quality signal.

**Key observations:**
- Ratios 10–15: operationally plausible (long, narrow service vessels)
- Ratios < 1: rare, may indicate swapped or misrecorded values
- VesselType = 0: single vessel isolated for inspection

---

No systemic structural anomalies. Dimension fields validated. Small outlier set documented. Full dataset usable for dimension-based analysis with edge cases noted.

In [51]:
display(
    AIS_01_01["Draft"]
    .describe()
)

count   5,289,721.00
mean            3.61
std             3.23
min             0.00
25%             2.00
50%             3.00
75%             4.60
max            25.50
Name: Draft, dtype: float64

In [52]:
draft_missing_mask = AIS_01_01["Draft"].isna()
draft_zero_mask = AIS_01_01["Draft"] == 0

draft_missing_or_zero_df = AIS_01_01.loc[
    draft_missing_mask | draft_zero_mask,
    metadata_cols
]

print("Draft inspection across all transmissions\n")

print("Rows with missing Draft:", draft_missing_mask.sum())
print("Rows with zero Draft:", draft_zero_mask.sum())
print("Rows with missing or zero Draft:", len(draft_missing_or_zero_df), "\n")

display(draft_missing_or_zero_df.head(30))

Draft inspection across all transmissions

Rows with missing Draft: 2006554
Rows with zero Draft: 1047931
Rows with missing or zero Draft: 3054485 



,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
0,PILOT BOAT SPRING PT,WDB8945,NaN,90.00,0.00,0.00,0.00,A
1,ALASKA CHALLENGER,WDH9586,IMO7938024,30.00,30.00,8.00,0.00,A
3,BART TURECAMO,WBR4464,IMO7338808,52.00,33.00,5.00,0.00,A
4,DOROTHY MORAN,WXU4654,IMO7716995,52.00,33.00,11.00,0.00,A
5,JAHAZI,NaN,IMO0000000,36.00,12.00,4.00,NaN,B
18,HENRY GIRLS,WDN9539,NaN,52.00,75.00,26.00,0.00,A
23,PARKSVILLE,CGKC,NaN,52.00,13.00,5.00,0.00,A
29,SAVANNAH,WDD9810,IMO9269738,52.00,28.00,10.00,0.00,A
38,MR SEAMAN,WCZ3468,IMO8964276,60.00,43.00,9.00,0.00,A
45,MAJESTIC,WDI7186,IMO8986016,60.00,50.00,10.00,0.00,A


In [53]:
draft_missing_or_zero_vessels_df = (
    AIS_01_01.loc[
        draft_missing_mask | draft_zero_mask,
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates(subset=["MMSI"])
    .sort_values("MMSI")
)

print("Draft inspection at vessel level\n")

print("Unique vessels with missing or zero Draft:", len(draft_missing_or_zero_vessels_df), "\n")

display(draft_missing_or_zero_vessels_df.head(30))

Draft inspection at vessel level

Unique vessels with missing or zero Draft: 8823 



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
36129,4061,BOOSTER 9000,0000000,IMO0000000,33.00,0.00,0.00,NaN,B
4919,3381234,ZEEPAARD,BO12345,IMO0000000,36.00,0.00,0.00,NaN,B
5657738,4556531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
5824125,4566545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
6094904,4566548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
2942,36926403,WARSHIP 25,NMNE,NaN,35.00,0.00,0.00,0.00,A
726,36968098,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,A
1268886,99043470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A
25072,103669999,MILADY,NaN,IMO0000000,37.00,17.00,5.00,NaN,B
284973,109070092,SNURREVAD,1,IMO0000000,30.00,2.00,2.00,NaN,B


In [54]:
usable_metadata_mask = (
    AIS_01_01["VesselType"].notna() &
    AIS_01_01["Length"].notna() &
    AIS_01_01["Width"].notna()
)


draft_missing_or_zero_with_metadata_df = (
    AIS_01_01.loc[
        (draft_missing_mask | draft_zero_mask) &
        usable_metadata_mask,
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates(subset=["MMSI"])
    .sort_values("MMSI")
)

total_affected_vessels = draft_missing_or_zero_vessels_df["MMSI"].nunique()
affected_with_metadata = draft_missing_or_zero_with_metadata_df["MMSI"].nunique()


print("Affected vessels (all):", total_affected_vessels)
print("Affected vessels with usable metadata:", affected_with_metadata)
print(
    "Reduction after filtering:",
    total_affected_vessels - affected_with_metadata
)
print(
    "Remaining share:",
    round((affected_with_metadata / total_affected_vessels) * 100, 2),
    "%",
    "\n"
)

display(draft_missing_or_zero_with_metadata_df.head(30))

Affected vessels (all): 8823
Affected vessels with usable metadata: 8765
Reduction after filtering: 58
Remaining share: 99.34 % 



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
36129,4061,BOOSTER 9000,0000000,IMO0000000,33.00,0.00,0.00,NaN,B
4919,3381234,ZEEPAARD,BO12345,IMO0000000,36.00,0.00,0.00,NaN,B
2942,36926403,WARSHIP 25,NMNE,NaN,35.00,0.00,0.00,0.00,A
726,36968098,US GOVERNMENT VESSEL,NaN,NaN,0.00,0.00,0.00,0.00,A
25072,103669999,MILADY,NaN,IMO0000000,37.00,17.00,5.00,NaN,B
284973,109070092,SNURREVAD,1,IMO0000000,30.00,2.00,2.00,NaN,B
47724,123456789,ODYSSEY,ODYSSEY,IMO0000000,37.00,25.00,10.00,0.00,B
190805,129129903,DRAGON FIRE,WCI3343,IMO0000000,36.00,16.00,5.00,NaN,B
5948414,155017179,0.C.D,CECE,IMO0000000,37.00,16.00,4.00,NaN,B
4567638,205402470,BELLA DONNA,OQ4024,IMO0000000,37.00,14.00,4.00,NaN,B


In [55]:
valid_dimension_mask = (
    (AIS_01_01["Length"] > 0) &
    (AIS_01_01["Width"] > 0) &
    AIS_01_01["Draft"].notna() &
    (AIS_01_01["Draft"] > 0)
)

draft_ratio_df = AIS_01_01.loc[
    valid_dimension_mask,
    ["MMSI"] + metadata_cols
].copy()

print("Draft ratio validation population\n")

print("Rows eligible for ratio analysis:", len(draft_ratio_df))
print("Unique vessels:", draft_ratio_df["MMSI"].nunique())

Draft ratio validation population

Rows eligible for ratio analysis: 4222865
Unique vessels: 6063


In [56]:
draft_ratio_df["draft_length_ratio"] = (
    draft_ratio_df["Draft"] /
    draft_ratio_df["Length"]
)

draft_ratio_df["draft_width_ratio"] = (
    draft_ratio_df["Draft"] /
    draft_ratio_df["Width"]
)

In [57]:
print("Draft-to-Length ratio summary\n")

display(
    draft_ratio_df["draft_length_ratio"]
    .describe()
    .round(3)
)

print("\nDraft-to-Width ratio summary\n")

display(
    draft_ratio_df["draft_width_ratio"]
    .describe()
    .round(3)
)

Draft-to-Length ratio summary



count   4,222,865.00
mean            0.09
std             0.08
min             0.00
25%             0.04
50%             0.07
75%             0.13
max             1.70
Name: draft_length_ratio, dtype: float64


Draft-to-Width ratio summary



count   4,222,865.00
mean            0.31
std             0.20
min             0.01
25%             0.21
50%             0.29
75%             0.38
max             4.25
Name: draft_width_ratio, dtype: float64

In [58]:
suspicious_draft_ratio_df = draft_ratio_df.loc[
    (
        draft_ratio_df["Draft"] >= draft_ratio_df["Length"]
    )
    |
    (
        draft_ratio_df["draft_length_ratio"] > 0.5
    )
    |
    (
        draft_ratio_df["Draft"] >= draft_ratio_df["Width"]
    )
]

affected_vessels = suspicious_draft_ratio_df["MMSI"].nunique()
total_vessels = draft_ratio_df["MMSI"].nunique()

affected_share = round(
    (affected_vessels / total_vessels) * 100,
    3
)

print("Draft plausibility check\n")

print("Affected vessels:", affected_vessels)
print("Total vessels evaluated:", total_vessels)
print("Affected share:", affected_share, "%")

Draft plausibility check

Affected vessels: 74
Total vessels evaluated: 6063
Affected share: 1.221 %


In [59]:
print("Suspicious Draft ratio vessels — inspection sample\n")

display(
    suspicious_draft_ratio_df[
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates(subset="MMSI")
    .sort_values("MMSI")
    .head(30)
)

Suspicious Draft ratio vessels — inspection sample



,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
319,303308000,AUTUMN DAWN,WDB8349,IMO8882234,30.00,43.00,10.00,10.00,A
1759,303612000,LEVIATHAN 1,WDL4265,IMO0000138,60.00,25.00,8.00,10.00,A
3742,312391000,ORION,V3DX3,IMO7708912,70.00,47.00,9.00,25.50,A
3097,316005713,PACIFIC FORCE,NaN,NaN,52.00,14.00,8.00,11.00,A
3993,316006562,OMNI COASTAL,VA9311,IMO6923084,52.00,26.00,8.00,14.00,A
6278329,316051489,S/Y VENTUROSA,CFY4040,NaN,36.00,33.00,8.00,13.00,A
239743,329010570,HIRUNDO,FAA5454,IMO9646003,71.00,40.00,10.00,11.00,A
3424791,338108506,VEDETTE,WSM7360,IMO6730073,50.00,11.00,2.00,2.00,A
5494375,338327000,DENIS SULLIVAN,WDA2917,IMO1100209,36.00,45.00,9.00,10.80,A
571564,341504000,MUTTY'S PRIDE,V4TL,IMO7917757,79.00,60.00,13.00,25.50,A


In [60]:
print("Suspicious Draft ratio vessels — numeric summary\n")

display(
    suspicious_draft_ratio_df[
        ["Length", "Width", "Draft"]
    ]
    .describe()
    .round(2)
)

Suspicious Draft ratio vessels — numeric summary



,Length,Width,Draft
count,"58,047.00","58,047.00","58,047.00"
mean,24.02,8.14,11.67
std,9.37,2.51,5.75
min,8.00,2.00,2.00
25%,17.00,7.00,9.00
50%,22.00,8.00,10.00
75%,28.00,9.00,12.90
max,83.00,18.00,25.50


In [61]:
print("Suspicious Draft ratio vessels — VesselType distribution\n")

display(
    suspicious_draft_ratio_df["VesselType"]
    .value_counts()
    .sort_index()
)

Suspicious Draft ratio vessels — VesselType distribution



VesselType
30.00     5801
31.00    22618
32.00     1588
35.00      683
36.00       25
37.00     1192
50.00      977
52.00    16412
57.00     1731
59.00     1110
60.00     3368
67.00      396
70.00     1230
71.00      423
79.00      493
Name: count, dtype: int64

In [62]:
print("Suspicious Draft ratio vessels — TransceiverClass distribution\n")

display(
    suspicious_draft_ratio_df["TransceiverClass"]
    .value_counts(dropna=False)
)

Suspicious Draft ratio vessels — TransceiverClass distribution



TransceiverClass
A    58047
Name: count, dtype: int64

### Draft Plausibility — Findings

Flagged vessels are mostly small-to-medium operational craft (tugboats, pilot boats). Median dimensions: Length ≈ 22m, Width ≈ 8m, Draft ≈ 10m.

These are designed to operate with relatively deep drafts — the ratios reflect vessel design rather than data errors.

VesselType distribution confirms this (Type 31, 52 dominant). All flagged vessels use TransceiverClass A.

---

Unusual ratios explained by vessel design. No data quality issue. All draft values kept. Field usable for operational analysis.

---

In [63]:
group_30_40_vessels = (
    AIS_01_01.loc[
        AIS_01_01["VesselType"].between(30, 40),
        "MMSI"
    ]
    .nunique()
)

total_unique_vessels = AIS_01_01["MMSI"].nunique()

group_30_40_share = round(
    (group_30_40_vessels / total_unique_vessels) * 100,
    2
)

print("Group 30–40 population context\n")

print("Unique vessels in group 30–40:", group_30_40_vessels)
print("Total unique vessels:", total_unique_vessels)
print("Share of total vessels:", group_30_40_share, "%")

Group 30–40 population context

Unique vessels in group 30–40: 9760
Total unique vessels: 14868
Share of total vessels: 65.64 %


### VesselType 30–40 Frequency Check

This group appears repeatedly across anomaly investigations. Tested whether it represents a true risk signal or reflects high baseline frequency.

---

Group 30–40 prevalence in anomaly sets is proportional to its population share. Not a quality signal — just a common vessel type. Analysis continues without type-specific exclusions.

---

---

## Position Validation

In [64]:
print("Latitude range:")
print("Min:", AIS_01_01["LAT"].min())
print("Max:", AIS_01_01["LAT"].max(), "\n")

print("Longitude range:")
print("Min:", AIS_01_01["LON"].min())
print("Max:", AIS_01_01["LON"].max(), "\n")

print("Missing LAT values:", AIS_01_01["LAT"].isna().sum())
print("Missing LON values:", AIS_01_01["LON"].isna().sum(), "\n")

print("Rows with LAT outside valid range:", ((AIS_01_01["LAT"] < -90) | (AIS_01_01["LAT"] > 90)).sum())
print("Rows with LON outside valid range:", ((AIS_01_01["LON"] < -180) | (AIS_01_01["LON"] > 180)).sum(), "\n")

print("Rows with LAT = 0:", (AIS_01_01["LAT"] == 0).sum())
print("Rows with LON = 0:", (AIS_01_01["LON"] == 0).sum())
print("Rows with LAT = 0 and LON = 0:", ((AIS_01_01["LAT"] == 0) & (AIS_01_01["LON"] == 0)).sum())

Latitude range:
Min: 0.11667
Max: 50.38428 

Longitude range:
Min: -160.08352
Max: 145.93094 

Missing LAT values: 0
Missing LON values: 0 

Rows with LAT outside valid range: 0
Rows with LON outside valid range: 0 

Rows with LAT = 0: 0
Rows with LON = 0: 0
Rows with LAT = 0 and LON = 0: 0


In [65]:
movement_df = (
    AIS_01_01[
        AIS_01_01["LAT"].notna() &
        AIS_01_01["LON"].notna()
    ]
    .copy()
)

movement_df = movement_df.sort_values(["MMSI", "BaseDateTime"])

In [66]:
movement_df["prev_time"] = movement_df.groupby("MMSI")["BaseDateTime"].shift(1)
movement_df["prev_lat"] = movement_df.groupby("MMSI")["LAT"].shift(1)
movement_df["prev_lon"] = movement_df.groupby("MMSI")["LON"].shift(1)
movement_df["prev_sog"] = movement_df.groupby("MMSI")["SOG"].shift(1)

movement_df["time_diff_minutes"] = (
    (movement_df["BaseDateTime"] - movement_df["prev_time"]).dt.total_seconds() / 60
)

movement_df["lat_diff"] = (movement_df["LAT"] - movement_df["prev_lat"]).abs()
movement_df["lon_diff"] = (movement_df["LON"] - movement_df["prev_lon"]).abs()
movement_df["coord_step_change"] = movement_df["lat_diff"] + movement_df["lon_diff"]

In [67]:
print("Time diff range (minutes):")
print("Min:", movement_df["time_diff_minutes"].min())
print("Max:", movement_df["time_diff_minutes"].max(), "\n")

print("Coordinate step change range:")
print("Min:", movement_df["coord_step_change"].min())
print("Max:", movement_df["coord_step_change"].max(), "\n")

print("Rows with non-positive time difference:", (movement_df["time_diff_minutes"] <= 0).sum())

Time diff range (minutes):
Min: 0.0
Max: 1397.5166666666667 

Coordinate step change range:
Min: 0.0
Max: 31.00765 

Rows with non-positive time difference: 258


In [68]:
suspicious_movement_mask = (
    (movement_df["time_diff_minutes"] > 0) &
    (
        (movement_df["coord_step_change"] > 5) |
        ((movement_df["SOG"] > 40) & (movement_df["coord_step_change"] < 0.01)) |
        ((movement_df["SOG"] == 0) & (movement_df["coord_step_change"] > 0.1))
    )
)

suspicious_movement_df = (
    movement_df.loc[
        suspicious_movement_mask,
        ["MMSI", "BaseDateTime", "prev_time", "LAT", "LON", "prev_lat", "prev_lon",
         "SOG", "prev_sog", "time_diff_minutes", "lat_diff", "lon_diff", "coord_step_change"] + metadata_cols
    ]
    .sort_values(["MMSI", "BaseDateTime"])
)

print(f"Number of suspicious movement rows: {len(suspicious_movement_df)}\n")

display(suspicious_movement_df.head(30))

Number of suspicious movement rows: 17453



,MMSI,BaseDateTime,prev_time,LAT,LON,prev_lat,prev_lon,SOG,prev_sog,time_diff_minutes,...,lon_diff,coord_step_change,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
3158014,210478000,2024-01-01 12:04:53,2024-01-01 12:03:44,18.65,-66.16,18.65,-66.16,102.30,8.40,1.15,...,0.00,0.00,TITAN UNIKUM,5BEV6,IMO9468437,81.00,153.00,20.00,8.20,A
6784182,220590000,2024-01-01 11:01:24,2024-01-01 10:58:24,48.42,-123.39,48.42,-123.39,102.30,0.00,3.00,...,0.00,0.00,MAERSK TENDER,OYGS2,IMO9388651,52.00,73.00,20.00,7.00,A
4360744,229074000,2024-01-01 16:35:32,2024-01-01 16:32:37,23.93,-73.69,18.33,-64.95,22.60,0.00,2.92,...,8.74,14.34,CELEBRITY REFLECTION,9HA3047,IMO9506459,60.00,319.00,48.00,8.60,A
4373092,229074000,2024-01-01 16:38:37,2024-01-01 16:35:32,18.33,-64.95,23.93,-73.69,0.00,22.60,3.08,...,8.74,14.34,CELEBRITY REFLECTION,9HA3047,IMO9506459,60.00,319.00,48.00,8.60,A
2504032,256003247,2024-01-01 01:51:23,2024-01-01 01:49:23,25.79,-80.16,25.79,-80.16,102.30,0.60,2.00,...,0.00,0.00,CIAO L,9HB9352,IMO0000000,37.00,17.00,6.00,NaN,B
506887,256003247,2024-01-01 01:54:23,2024-01-01 01:52:53,25.79,-80.16,25.79,-80.16,102.30,1.40,1.50,...,0.00,0.00,CIAO L,9HB9352,IMO0000000,37.00,17.00,6.00,NaN,B
496667,256003247,2024-01-01 01:56:54,2024-01-01 01:54:23,25.79,-80.16,25.79,-80.16,102.30,102.30,2.52,...,0.00,0.00,CIAO L,9HB9352,IMO0000000,37.00,17.00,6.00,NaN,B
534886,256003247,2024-01-01 02:02:52,2024-01-01 01:56:54,25.79,-80.16,25.79,-80.16,102.30,102.30,5.97,...,0.00,0.00,CIAO L,9HB9352,IMO0000000,37.00,17.00,6.00,NaN,B
723609,256003247,2024-01-01 02:08:53,2024-01-01 02:02:52,25.79,-80.16,25.79,-80.16,102.30,102.30,6.02,...,0.00,0.00,CIAO L,9HB9352,IMO0000000,37.00,17.00,6.00,NaN,B
582138,256003247,2024-01-01 02:11:54,2024-01-01 02:08:53,25.79,-80.16,25.79,-80.16,102.30,102.30,3.02,...,0.00,0.00,CIAO L,9HB9352,IMO0000000,37.00,17.00,6.00,NaN,B


In [69]:
jump_suspicious_mask = (
    (movement_df["time_diff_minutes"] > 0) &
    (movement_df["time_diff_minutes"] <= 5) &
    (movement_df["coord_step_change"] > 1)
)

jump_suspicious_df = (
    movement_df.loc[
        jump_suspicious_mask,
        ["MMSI", "BaseDateTime", "prev_time", "LAT", "LON", "prev_lat", "prev_lon",
         "SOG", "prev_sog", "time_diff_minutes", "lat_diff", "lon_diff", "coord_step_change"] + metadata_cols
    ]
    .sort_values(["MMSI", "BaseDateTime"])
)

print(f"Number of suspicious jump rows: {len(jump_suspicious_df)}\n")

display(jump_suspicious_df.head(30))

Number of suspicious jump rows: 965



,MMSI,BaseDateTime,prev_time,LAT,LON,prev_lat,prev_lon,SOG,prev_sog,time_diff_minutes,...,lon_diff,coord_step_change,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
4360744,229074000,2024-01-01 16:35:32,2024-01-01 16:32:37,23.93,-73.69,18.33,-64.95,22.60,0.00,2.92,...,8.74,14.34,CELEBRITY REFLECTION,9HA3047,IMO9506459,60.00,319.00,48.00,8.60,A
4373092,229074000,2024-01-01 16:38:37,2024-01-01 16:35:32,18.33,-64.95,23.93,-73.69,0.00,22.60,3.08,...,8.74,14.34,CELEBRITY REFLECTION,9HA3047,IMO9506459,60.00,319.00,48.00,8.60,A
6950470,310000000,2024-01-01 15:23:17,2024-01-01 15:19:09,27.15,-79.55,25.85,-80.16,37.70,0.00,4.13,...,0.61,1.91,UNFORGETTABLEOF,ZCOU9,IMO0000000,37.00,0.00,0.00,NaN,B
179190,311001094,2024-01-01 00:17:05,2024-01-01 00:16:02,32.92,-76.48,38.87,-76.40,22.40,18.30,1.05,...,0.08,6.04,CARNIVAL LEGEND,C6FF2,IMO9224726,60.00,293.00,40.00,0.00,A
80328,311001094,2024-01-01 00:18:08,2024-01-01 00:17:05,38.86,-76.40,32.92,-76.48,18.20,22.40,1.05,...,0.08,6.02,CARNIVAL LEGEND,C6FF2,IMO9224726,60.00,293.00,40.00,0.00,A
1722128,311001094,2024-01-01 06:32:30,2024-01-01 06:31:26,30.63,-77.77,37.09,-76.11,21.30,16.90,1.07,...,1.66,8.12,CARNIVAL LEGEND,C6FF2,IMO9224726,60.00,293.00,40.00,0.00,A
1726634,311001094,2024-01-01 06:33:39,2024-01-01 06:32:30,37.08,-76.11,30.63,-77.77,16.90,21.30,1.15,...,1.66,8.12,CARNIVAL LEGEND,C6FF2,IMO9224726,60.00,293.00,40.00,0.00,A
1930055,311001094,2024-01-01 07:15:45,2024-01-01 07:14:42,35.37,-75.15,36.94,-75.97,20.50,9.80,1.05,...,0.83,2.40,CARNIVAL LEGEND,C6FF2,IMO9224726,60.00,293.00,40.00,0.00,A
6645545,311001094,2024-01-01 07:16:52,2024-01-01 07:15:45,36.94,-75.97,35.37,-75.15,9.80,20.50,1.12,...,0.82,2.39,CARNIVAL LEGEND,C6FF2,IMO9224726,60.00,293.00,40.00,0.00,A
6989925,311027100,2024-01-01 16:26:35,2024-01-01 16:25:32,21.95,-159.36,21.30,-157.87,1.20,0.00,1.05,...,1.49,2.14,SEABOURN SOJOURN,C6YA5,IMO9417098,69.00,198.00,28.00,6.60,A


In [70]:
movement_display_cols = [
    "MMSI",
    "VesselName",
    "CallSign",
    "BaseDateTime",
    "prev_time",
    "LAT",
    "LON",
    "prev_lat",
    "prev_lon",
    "SOG",
    "prev_sog",
    "time_diff_minutes",
    "lat_diff",
    "lon_diff",
    "coord_step_change"
]
SHORT_TIME_THRESHOLD = 5
LARGE_MOVE_THRESHOLD = 1

teleport_mask = (
    (movement_df["time_diff_minutes"] > 0) &
    (movement_df["time_diff_minutes"] <= SHORT_TIME_THRESHOLD) &
    (movement_df["coord_step_change"] > LARGE_MOVE_THRESHOLD)
)

teleport_df = (
    movement_df.loc[teleport_mask, movement_display_cols]
    .sort_values(["MMSI", "BaseDateTime"])
)

print(f"Teleportation rows: {len(teleport_df)}")

display(teleport_df.head(30))

Teleportation rows: 965


,MMSI,VesselName,CallSign,BaseDateTime,prev_time,LAT,LON,prev_lat,prev_lon,SOG,prev_sog,time_diff_minutes,lat_diff,lon_diff,coord_step_change
4360744,229074000,CELEBRITY REFLECTION,9HA3047,2024-01-01 16:35:32,2024-01-01 16:32:37,23.93,-73.69,18.33,-64.95,22.60,0.00,2.92,5.60,8.74,14.34
4373092,229074000,CELEBRITY REFLECTION,9HA3047,2024-01-01 16:38:37,2024-01-01 16:35:32,18.33,-64.95,23.93,-73.69,0.00,22.60,3.08,5.60,8.74,14.34
6950470,310000000,UNFORGETTABLEOF,ZCOU9,2024-01-01 15:23:17,2024-01-01 15:19:09,27.15,-79.55,25.85,-80.16,37.70,0.00,4.13,1.30,0.61,1.91
179190,311001094,CARNIVAL LEGEND,C6FF2,2024-01-01 00:17:05,2024-01-01 00:16:02,32.92,-76.48,38.87,-76.40,22.40,18.30,1.05,5.95,0.08,6.04
80328,311001094,CARNIVAL LEGEND,C6FF2,2024-01-01 00:18:08,2024-01-01 00:17:05,38.86,-76.40,32.92,-76.48,18.20,22.40,1.05,5.94,0.08,6.02
1722128,311001094,CARNIVAL LEGEND,C6FF2,2024-01-01 06:32:30,2024-01-01 06:31:26,30.63,-77.77,37.09,-76.11,21.30,16.90,1.07,6.46,1.66,8.12
1726634,311001094,CARNIVAL LEGEND,C6FF2,2024-01-01 06:33:39,2024-01-01 06:32:30,37.08,-76.11,30.63,-77.77,16.90,21.30,1.15,6.45,1.66,8.12
1930055,311001094,CARNIVAL LEGEND,C6FF2,2024-01-01 07:15:45,2024-01-01 07:14:42,35.37,-75.15,36.94,-75.97,20.50,9.80,1.05,1.57,0.83,2.40
6645545,311001094,CARNIVAL LEGEND,C6FF2,2024-01-01 07:16:52,2024-01-01 07:15:45,36.94,-75.97,35.37,-75.15,9.80,20.50,1.12,1.57,0.82,2.39
6989925,311027100,SEABOURN SOJOURN,C6YA5,2024-01-01 16:26:35,2024-01-01 16:25:32,21.95,-159.36,21.30,-157.87,1.20,0.00,1.05,0.65,1.49,2.14


In [71]:
unique_vessels = teleport_df["MMSI"].nunique()

print("Unique vessels in teleportation set:", unique_vessels)

Unique vessels in teleportation set: 32


In [72]:
vessel_counts = (
    teleport_df.groupby(["MMSI", "VesselName"])
    .size()
    .reset_index(name="suspicious_events")
    .sort_values("suspicious_events", ascending=False)
)

display(vessel_counts.head(20))

,MMSI,VesselName,suspicious_events
18,367373000,WD143 SHELL RIG,463
13,338393461,JUST A SPLASH,167
17,367075320,MACY LYNN,91
16,360000000,INGRAM DUPO,86
23,367761750,TRADITION,74
6,338056923,EPIPHANY,12
24,367799460,MAJESTIK,9
21,367662790,LADY JANET,8
2,311001094,CARNIVAL LEGEND,6
11,338227769,BUCKET LIST,5


In [73]:
SIGNIFICANT_EVENT_THRESHOLD = 10

significant_vessels = vessel_counts[
    vessel_counts["suspicious_events"] > SIGNIFICANT_EVENT_THRESHOLD
]

display(significant_vessels)

,MMSI,VesselName,suspicious_events
18,367373000,WD143 SHELL RIG,463
13,338393461,JUST A SPLASH,167
17,367075320,MACY LYNN,91
16,360000000,INGRAM DUPO,86
23,367761750,TRADITION,74
6,338056923,EPIPHANY,12


In [74]:
significant_mmsi = significant_vessels["MMSI"].unique()

significant_metadata_df = (
    movement_df.loc[
        movement_df["MMSI"].isin(significant_mmsi),
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates(subset=["MMSI"])
    .sort_values("MMSI")
)

display(significant_metadata_df)

,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
4572,338056923,EPIPHANY,NaN,NaN,37.00,21.00,6.00,2.00,A
6156,338393461,JUST A SPLASH,NaN,IMO0000000,37.00,14.00,6.00,NaN,B
290,360000000,INGRAM DUPO,0000000,NaN,90.00,52.00,14.00,2.00,A
941,367075320,MACY LYNN,WDC7693,IMO100555451,31.00,81.00,34.00,2.50,A
1704,367373000,WD143 SHELL RIG,WDE5894,NaN,37.00,100.00,100.00,0.00,A
6582940,367761750,TRADITION,WDJ2790,IMO0000000,30.00,17.00,6.00,NaN,B


In [75]:
teleport_with_type_df = (
    teleport_df.merge(
        AIS_01_01[["MMSI", "VesselType"]].drop_duplicates(subset=["MMSI"]),
        on="MMSI",
        how="left"
    )
)

vessel_type_impact = (
    teleport_with_type_df
    .groupby("VesselType")
    .size()
    .reset_index(name="suspicious_events")
    .sort_values("suspicious_events", ascending=False)
)

display(vessel_type_impact)

,VesselType,suspicious_events
4,37.00,680
2,31.00,93
10,90.00,86
1,30.00,75
5,60.00,10
3,36.00,9
6,69.00,4
9,74.00,3
8,71.00,2
7,70.00,2


In [76]:
movement_df["movement_rate"] = (
    movement_df["coord_step_change"] /
    movement_df["time_diff_minutes"]
)

display(
    movement_df["movement_rate"]
    .describe()
)

C:\Users\kados\anaconda3\envs\main\Lib\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count   7,281,192.00
mean             inf
std              NaN
min             0.00
25%             0.00
50%             0.00
75%             0.00
max              inf
Name: movement_rate, dtype: float64

In [77]:
print(
    "Rows with zero time difference:",
    (movement_df["time_diff_minutes"] == 0).sum()
)

Rows with zero time difference: 258


In [78]:
non_positive_time_rows = (movement_df["time_diff_minutes"] <= 0).sum()
zero_time_rows = (movement_df["time_diff_minutes"] == 0).sum()
negative_time_rows = (movement_df["time_diff_minutes"] < 0).sum()

print("Rows with non-positive time difference:", non_positive_time_rows)
print("Rows with zero time difference:", zero_time_rows)
print("Rows with negative time difference:", negative_time_rows)

Rows with non-positive time difference: 258
Rows with zero time difference: 258
Rows with negative time difference: 0


In [79]:
print(
    "Are all non-positive time differences exactly zero?",
    non_positive_time_rows == zero_time_rows
)

Are all non-positive time differences exactly zero? True


In [80]:
zero_time_df = (
    movement_df.loc[
        movement_df["time_diff_minutes"] == 0,
        movement_display_cols
    ]
    .sort_values(["MMSI", "BaseDateTime"])
)

display(zero_time_df.head(30))

,MMSI,VesselName,CallSign,BaseDateTime,prev_time,LAT,LON,prev_lat,prev_lon,SOG,prev_sog,time_diff_minutes,lat_diff,lon_diff,coord_step_change
6600357,229081000,ATLANTIC ELM,9HA3054,2024-01-01 06:00:00,2024-01-01 06:00:00,17.69,-66.70,17.69,-66.70,0.50,0.50,0.00,0.00,0.00,0.00
1273332,229830000,RIO,9HA5610,2024-01-01 03:59:59,2024-01-01 03:59:59,18.35,-64.58,18.35,-64.58,0.00,0.00,0.00,0.00,0.00,0.00
1083310,232010350,GLENPARK,MBQB5,2024-01-01 04:07:00,2024-01-01 04:07:00,47.27,-122.44,47.27,-122.44,0.10,0.00,0.00,0.00,0.00,0.00
3743381,232010350,GLENPARK,MBQB5,2024-01-01 14:13:00,2024-01-01 14:13:00,47.27,-122.44,47.28,-122.44,0.20,0.10,0.00,0.00,0.00,0.01
7035963,232010350,GLENPARK,MBQB5,2024-01-01 17:25:00,2024-01-01 17:25:00,47.28,-122.44,47.27,-122.44,0.00,0.00,0.00,0.00,0.00,0.00
5416641,232010350,GLENPARK,MBQB5,2024-01-01 20:19:00,2024-01-01 20:19:00,47.27,-122.44,47.28,-122.44,0.10,0.10,0.00,0.00,0.00,0.00
5516377,232010350,GLENPARK,MBQB5,2024-01-01 20:43:00,2024-01-01 20:43:00,47.27,-122.44,47.28,-122.44,0.10,0.20,0.00,0.00,0.00,0.00
6082736,232010350,GLENPARK,MBQB5,2024-01-01 22:49:00,2024-01-01 22:49:00,47.27,-122.44,47.27,-122.44,0.00,0.20,0.00,0.00,0.00,0.00
1329651,240975000,KIMOLOS,SVAV7,2024-01-01 04:59:59,2024-01-01 04:59:59,29.10,-94.57,29.10,-94.57,0.00,0.00,0.00,0.00,0.00,0.00
2097804,240975000,KIMOLOS,SVAV7,2024-01-01 07:59:59,2024-01-01 07:59:59,29.10,-94.57,29.10,-94.57,0.00,0.00,0.00,0.00,0.00,0.00


In [81]:
zero_time_vessel_counts = (
    zero_time_df.groupby(["MMSI", "VesselName"])
    .size()
    .reset_index(name="zero_time_events")
    .sort_values("zero_time_events", ascending=False)
)

display(zero_time_vessel_counts.head(20))

,MMSI,VesselName,zero_time_events
2,232010350,GLENPARK,6
3,240975000,KIMOLOS,5
45,338456187,LATITUDES,5
141,367767350,TROPIC THUNDER,5
31,338206398,ROSIE,4
108,367513170,LASSEN,4
166,368260290,HONU KAI,4
56,357777000,BUENA VENTURA,3
171,368784000,LIBERTY PEACE,3
80,367089490,BLACK SWAN,3


In [82]:
ZERO_SPEED_THRESHOLD = 0.5
STATIONARY_MOVE_THRESHOLD = 0.5

stationary_flip_mask = (
    (movement_df["SOG"] <= ZERO_SPEED_THRESHOLD) &
    (movement_df["coord_step_change"] > STATIONARY_MOVE_THRESHOLD)
)

stationary_flip_df = (
    movement_df.loc[stationary_flip_mask, movement_display_cols]
    .sort_values(["MMSI", "BaseDateTime"])
)

print(f"Stationary movement rows: {len(stationary_flip_df)}")

display(stationary_flip_df.head(30))

Stationary movement rows: 870


,MMSI,VesselName,CallSign,BaseDateTime,prev_time,LAT,LON,prev_lat,prev_lon,SOG,prev_sog,time_diff_minutes,lat_diff,lon_diff,coord_step_change
4373092,229074000,CELEBRITY REFLECTION,9HA3047,2024-01-01 16:38:37,2024-01-01 16:35:32,18.33,-64.95,23.93,-73.69,0.00,22.60,3.08,5.60,8.74,14.34
7275343,303458000,BRAD DARTEZ,WDH4077,2024-01-01 23:45:15,2024-01-01 03:05:54,27.65,-92.41,28.48,-90.82,0.10,8.40,"1,239.35",0.82,1.59,2.41
4316240,310000000,UNFORGETTABLEOF,ZCOU9,2024-01-01 16:24:48,2024-01-01 15:26:16,25.85,-80.16,27.19,-79.53,0.00,0.70,58.53,1.34,0.63,1.98
2757582,311000969,MARGARITAVILLE A.S.P,C6EQ3,2024-01-01 10:30:33,2024-01-01 02:03:13,26.45,-78.88,26.86,-79.47,0.30,7.30,507.33,0.42,0.59,1.01
879220,311027100,SEABOURN SOJOURN,C6YA5,2024-01-01 03:13:32,2024-01-01 03:06:33,21.30,-157.87,20.64,-157.54,0.00,14.00,6.98,0.67,0.32,0.99
4325361,311027100,SEABOURN SOJOURN,C6YA5,2024-01-01 16:28:30,2024-01-01 16:26:35,21.30,-157.87,21.95,-159.36,0.00,1.20,1.92,0.65,1.49,2.14
5678944,311027100,SEABOURN SOJOURN,C6YA5,2024-01-01 21:17:37,2024-01-01 21:16:32,21.95,-159.36,21.30,-157.87,0.00,0.00,1.08,0.65,1.49,2.14
5683966,311027100,SEABOURN SOJOURN,C6YA5,2024-01-01 21:19:31,2024-01-01 21:17:37,21.30,-157.87,21.95,-159.36,0.00,0.00,1.90,0.65,1.49,2.14
2080012,316029941,PELA,NaN,2024-01-01 07:52:04,2024-01-01 07:49:35,49.31,-123.09,48.95,-124.42,0.20,19.70,2.48,0.36,1.33,1.70
6959033,338032000,AGNES CANDIES,KAGK,2024-01-01 15:40:06,2024-01-01 02:28:37,28.17,-89.22,28.38,-88.55,0.00,9.20,791.48,0.21,0.67,0.88


In [83]:
ZERO_TIME_MOVE_THRESHOLD = 0.05

zero_time_large_move_df = movement_df[
    (movement_df["time_diff_minutes"] == 0) &
    (movement_df["coord_step_change"] > ZERO_TIME_MOVE_THRESHOLD)
]

print(
    "Zero-time rows with meaningful movement:",
    len(zero_time_large_move_df)
)

display(
    zero_time_large_move_df
    .sort_values("coord_step_change", ascending=False)
    .head(30)
)

Zero-time rows with meaningful movement: 1


,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,...,Time,prev_time,prev_lat,prev_lon,prev_sog,time_diff_minutes,lat_diff,lon_diff,coord_step_change,movement_rate
3872492,367313160,2024-01-01 12:42:16,47.66,-122.38,0.00,267.70,511.00,SAN JUAN ENTERPRISE,NaN,WDD9563,...,12:42:16,2024-01-01 12:42:16,47.63,-122.25,0.20,0.00,0.03,0.13,0.16,inf


In [84]:
ciao_df = (
    movement_df.loc[
        movement_df["VesselName"] == "CIAO L",
        movement_display_cols
    ]
    .sort_values(["MMSI", "BaseDateTime"])
)

print(f"CIAO L rows: {len(ciao_df)}")

display(ciao_df)

CIAO L rows: 474


,MMSI,VesselName,CallSign,BaseDateTime,prev_time,LAT,LON,prev_lat,prev_lon,SOG,prev_sog,time_diff_minutes,lat_diff,lon_diff,coord_step_change
590474,256003247,CIAO L,9HB9352,2024-01-01 01:49:23,NaT,25.79,-80.16,NaN,NaN,0.60,NaN,NaN,NaN,NaN,NaN
2504032,256003247,CIAO L,9HB9352,2024-01-01 01:51:23,2024-01-01 01:49:23,25.79,-80.16,25.79,-80.16,102.30,0.60,2.00,0.00,0.00,0.00
499509,256003247,CIAO L,9HB9352,2024-01-01 01:52:53,2024-01-01 01:51:23,25.79,-80.16,25.79,-80.16,1.40,102.30,1.50,0.00,0.00,0.00
506887,256003247,CIAO L,9HB9352,2024-01-01 01:54:23,2024-01-01 01:52:53,25.79,-80.16,25.79,-80.16,102.30,1.40,1.50,0.00,0.00,0.00
496667,256003247,CIAO L,9HB9352,2024-01-01 01:56:54,2024-01-01 01:54:23,25.79,-80.16,25.79,-80.16,102.30,102.30,2.52,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6360754,256003247,CIAO L,9HB9352,2024-01-01 23:51:47,2024-01-01 23:50:17,25.79,-80.16,25.79,-80.16,1.10,1.70,1.50,0.00,0.00,0.00
6363365,256003247,CIAO L,9HB9352,2024-01-01 23:53:47,2024-01-01 23:51:47,25.79,-80.16,25.79,-80.16,0.20,1.10,2.00,0.00,0.00,0.00
6408813,256003247,CIAO L,9HB9352,2024-01-01 23:55:47,2024-01-01 23:53:47,25.79,-80.16,25.79,-80.16,1.20,0.20,2.00,0.00,0.00,0.00
7283638,256003247,CIAO L,9HB9352,2024-01-01 23:57:48,2024-01-01 23:55:47,25.79,-80.16,25.79,-80.16,1.10,1.20,2.02,0.00,0.00,0.00


### Position Validation — Summary

All coordinates within valid geographic ranges (-90/90 LAT, -180/180 LON).

Movement step analysis confirms expected vessel motion patterns. No systemic teleportation.

Zero-time intervals attributed to duplicate transmissions, not position errors.

---

Position fields structurally valid. Approved for trajectory analysis. All LAT/LON values retained. Zero-time duplicates available for deduplication during curation.

---

## Movement Signal Validation

In [85]:
print("Speed Over Ground (SOG) — summary\n")

display(
    AIS_01_01["SOG"]
    .describe()
    .round(2)
)

print("\nSOG range")

print("Min:", AIS_01_01["SOG"].min())
print("Max:", AIS_01_01["SOG"].max())

Speed Over Ground (SOG) — summary



count   7,296,275.00
mean            2.01
std             6.23
min             0.00
25%             0.00
50%             0.00
75%             0.30
max           102.30
Name: SOG, dtype: float64


SOG range
Min: 0.0
Max: 102.3


In [86]:
negative_sog = AIS_01_01["SOG"] < 0

print("Negative SOG values:",
      negative_sog.sum())

Negative SOG values: 0


In [87]:
extreme_sog = AIS_01_01["SOG"] > 100

print("SOG > 100 knots:",
      extreme_sog.sum())

SOG > 100 knots: 16633


In [88]:
print("Extreme speed transmissions (SOG > 100 knots)\n")

display(
    AIS_01_01.loc[
        extreme_sog,
        ["MMSI"] + movement_cols
    ]
    .sort_values("SOG", ascending=False)
    .head(30)
)

Extreme speed transmissions (SOG > 100 knots)



,MMSI,BaseDateTime,Date,Time,LAT,LON,SOG,COG,Heading
7284956,368225880,2024-01-01 23:57:55,2024-01-01,23:57:55,34.72,-76.71,102.30,360.00,511.00
1033,368360000,2024-01-01 00:00:04,2024-01-01,00:00:04,29.12,-90.21,102.30,360.00,511.00
1083,367001790,2024-01-01 00:00:04,2024-01-01,00:00:04,47.98,-122.22,102.30,360.00,36.00
1096,368228180,2024-01-01 00:00:03,2024-01-01,00:00:03,41.72,-87.54,102.30,360.00,511.00
1833,367400840,2024-01-01 00:00:07,2024-01-01,00:00:07,30.36,-88.51,102.30,360.00,90.00
1889,367659880,2024-01-01 00:00:08,2024-01-01,00:00:08,28.89,-90.00,102.30,360.00,511.00
1941,367087240,2024-01-01 00:00:02,2024-01-01,00:00:02,41.17,-73.18,102.30,360.00,511.00
1999,367090580,2024-01-01 00:00:04,2024-01-01,00:00:04,38.94,-75.32,102.30,360.00,511.00
2457,367482320,2024-01-01 00:00:07,2024-01-01,00:00:07,18.01,-66.76,102.30,360.00,511.00
7276710,367400840,2024-01-01 23:47:38,2024-01-01,23:47:38,30.36,-88.51,102.30,360.00,91.00


In [89]:
print("Unique values — movement fields\n")

display(
    AIS_01_01[["SOG", "COG", "Heading"]]
    .nunique()
    .rename("unique_count")
)

print("\nSample unique values per column\n")

for col in ["SOG", "COG", "Heading"]:
    print(f"\n{col} — first 20 unique values:")
    
    display(
        sorted(
            AIS_01_01[col]
            .dropna()
            .unique()
        )[:20]
    )

Unique values — movement fields



SOG         548
COG        3601
Heading     364
Name: unique_count, dtype: int64


Sample unique values per column


SOG — first 20 unique values:


[np.float64(0.0),
 np.float64(0.1),
 np.float64(0.2),
 np.float64(0.3),
 np.float64(0.4),
 np.float64(0.5),
 np.float64(0.6),
 np.float64(0.7),
 np.float64(0.8),
 np.float64(0.9),
 np.float64(1.0),
 np.float64(1.1),
 np.float64(1.2),
 np.float64(1.3),
 np.float64(1.4),
 np.float64(1.5),
 np.float64(1.6),
 np.float64(1.7),
 np.float64(1.8),
 np.float64(1.9)]


COG — first 20 unique values:


[np.float64(0.0),
 np.float64(0.1),
 np.float64(0.2),
 np.float64(0.3),
 np.float64(0.4),
 np.float64(0.5),
 np.float64(0.6),
 np.float64(0.7),
 np.float64(0.8),
 np.float64(0.9),
 np.float64(1.0),
 np.float64(1.1),
 np.float64(1.2),
 np.float64(1.3),
 np.float64(1.4),
 np.float64(1.5),
 np.float64(1.6),
 np.float64(1.7),
 np.float64(1.8),
 np.float64(1.9)]


Heading — first 20 unique values:


[np.float64(0.0),
 np.float64(1.0),
 np.float64(2.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(5.0),
 np.float64(6.0),
 np.float64(7.0),
 np.float64(8.0),
 np.float64(9.0),
 np.float64(10.0),
 np.float64(11.0),
 np.float64(12.0),
 np.float64(13.0),
 np.float64(14.0),
 np.float64(15.0),
 np.float64(16.0),
 np.float64(17.0),
 np.float64(18.0),
 np.float64(19.0)]

In [90]:
print("Sentinel pattern verification within extreme SOG rows\n")

extreme_mask = AIS_01_01["SOG"] > 100

sentinel_pattern_mask = (
    (AIS_01_01["SOG"] == 102.3) &
    (AIS_01_01["COG"] == 360) &
    (AIS_01_01["Heading"] == 511)
)

extreme_rows = extreme_mask.sum()

sentinel_rows = (
    extreme_mask &
    sentinel_pattern_mask
).sum()

non_sentinel_rows = extreme_rows - sentinel_rows

print("Total extreme SOG rows:", extreme_rows)
print("Rows matching sentinel pattern:", sentinel_rows)
print("Rows NOT matching sentinel pattern:", non_sentinel_rows)

Sentinel pattern verification within extreme SOG rows

Total extreme SOG rows: 16633
Rows matching sentinel pattern: 8267
Rows NOT matching sentinel pattern: 8366


In [91]:
print("Unique vessels — sentinel placeholder pattern\n")

sentinel_vessels_df = (
    AIS_01_01.loc[
        sentinel_pattern_mask,
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates(subset="MMSI")
    .sort_values("MMSI")
)

print(
    "Unique vessels with placeholder pattern:",
    sentinel_vessels_df["MMSI"].nunique()
)

display(
    sentinel_vessels_df.head(30)
)

Unique vessels — sentinel placeholder pattern

Unique vessels with placeholder pattern: 117


,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
496667,256003247,CIAO L,9HB9352,IMO0000000,37.00,17.00,6.00,NaN,B
3230481,316002306,WINDWARD II,NaN,IMO6417047,30.00,15.00,4.00,NaN,B
672316,316014014,TAKAYA,VG2773,NaN,53.00,14.00,5.00,1.40,A
2357592,316024235,HELEN J,CFN6439,NaN,31.00,15.00,6.00,3.60,A
2757722,316027733,VFPA4,CFK5368,NaN,53.00,13.00,3.00,1.80,A
4644126,316029221,BALLADEER I,NaN,IMO0000000,37.00,21.00,6.00,NaN,B
5358364,316029689,KIWI DREAM,NaN,IMO0000000,36.00,11.00,4.00,NaN,B
2507836,316039229,REPOSADO,CFA2949,IMO0000000,37.00,31.00,8.00,NaN,B
6214870,316047506,RAGTIME GAL,CHA2061,IMO0000000,36.00,14.00,4.00,NaN,B
5004350,316047517,POLONAISE,NaN,IMO0000000,36.00,10.00,2.00,NaN,B


In [92]:
print("\nTransmission vs vessel footprint\n")

print(
    "Total placeholder transmissions:",
    sentinel_pattern_mask.sum()
)

print(
    "Unique vessels involved:",
    sentinel_vessels_df["MMSI"].nunique()
)


Transmission vs vessel footprint

Total placeholder transmissions: 8267
Unique vessels involved: 117


In [93]:
print("Unique vessels — non-sentinel extreme SOG cases\n")

non_sentinel_extreme_mask = (
    extreme_mask &
    ~sentinel_pattern_mask
)

non_sentinel_vessels_df = (
    AIS_01_01.loc[
        non_sentinel_extreme_mask,
        ["MMSI"] + metadata_cols
    ]
    .drop_duplicates(subset="MMSI")
    .sort_values("MMSI")
)

print(
    "Unique vessels with non-sentinel extreme speeds:",
    non_sentinel_vessels_df["MMSI"].nunique()
)

display(
    non_sentinel_vessels_df.head(30)
)

Unique vessels — non-sentinel extreme SOG cases

Unique vessels with non-sentinel extreme speeds: 155


,MMSI,VesselName,CallSign,IMO,VesselType,Length,Width,Draft,TransceiverClass
3158014,210478000,TITAN UNIKUM,5BEV6,IMO9468437,81.00,153.00,20.00,8.20,A
6784182,220590000,MAERSK TENDER,OYGS2,IMO9388651,52.00,73.00,20.00,7.00,A
1926680,256509000,OREOS,9HA5866,IMO9381380,37.00,119.00,10.00,NaN,B
2944171,303390000,RN WEEKS,WBV3250,IMO8516079,33.00,90.00,18.00,6.10,A
2458440,303465000,SNOHOMISH,WDB9022,IMO7227384,32.00,31.00,9.00,3.10,A
2010692,303681000,MOUNT DRUM,WDJ7294,IMO9830460,32.00,34.00,11.00,0.00,A
3787472,316001940,FLORENCE SPIRIT,CFCB,IMO9314600,70.00,146.00,21.00,7.50,A
196830,316024256,SHARON M I,CFP2006,IMO9084059,52.00,35.00,11.00,5.20,A
2619059,316033399,SST SALISH,CFCP,IMO9744489,31.00,20.00,10.00,0.00,A
1116839,319064900,OKTO,ZGDZ2,IMO1012335,37.00,66.00,10.00,3.30,A


### SOG Validation — Summary

16,633 transmissions with SOG > 100 knots identified.

~Half match the known AIS placeholder pattern (SOG = 102.3, COG = 360, Heading = 511) — confirms "not available" rather than actual movement.

Remaining high-speed records distributed across multiple vessels with valid metadata. No concentration or structural anomalies.

---

Movement fields structurally valid. Sentinel values confirmed. SOG = 102.3 treated as null in speed calculations. All other values retained.

### COG and Heading Validation

In [94]:
print("COG and Heading validation — range and structural checks\n")

cog_negative = (AIS_01_01["COG"] < 0)
cog_over_360 = (AIS_01_01["COG"] > 360)

heading_negative = (AIS_01_01["Heading"] < 0)
heading_over_360 = (AIS_01_01["Heading"] > 360)
heading_equal_511 = (AIS_01_01["Heading"] == 511)

print("COG negative values:", cog_negative.sum())
print("COG > 360:", cog_over_360.sum())

print("\nHeading negative values:", heading_negative.sum())
print("Heading > 360:", heading_over_360.sum())
print("Heading = 511 (sentinel):", heading_equal_511.sum())

COG and Heading validation — range and structural checks

COG negative values: 0
COG > 360: 0

Heading negative values: 0
Heading > 360: 3827376
Heading = 511 (sentinel): 3827373


In [95]:
print("Sample — Heading sentinel values (511)\n")

display(
    AIS_01_01.loc[
        AIS_01_01["Heading"] == 511,
        movement_cols
    ]
    .head(20)
)

Sample — Heading sentinel values (511)



,BaseDateTime,Date,Time,LAT,LON,SOG,COG,Heading
0,2024-01-01 00:00:03,2024-01-01,00:00:03,43.65,-70.25,0.00,358.80,511.00
3,2024-01-01 00:00:05,2024-01-01,00:00:05,39.89,-75.18,0.00,304.40,511.00
4,2024-01-01 00:00:06,2024-01-01,00:00:06,18.33,-64.95,0.00,332.60,511.00
5,2024-01-01 00:00:05,2024-01-01,00:00:05,38.96,-76.48,0.10,111.30,511.00
6,2024-01-01 00:00:02,2024-01-01,00:00:02,33.75,-118.22,0.00,281.80,511.00
9,2024-01-01 00:00:04,2024-01-01,00:00:04,29.97,-90.24,0.00,144.30,511.00
12,2024-01-01 00:00:03,2024-01-01,00:00:03,30.53,-88.12,0.00,306.20,511.00
13,2024-01-01 00:00:06,2024-01-01,00:00:06,27.84,-97.07,1.40,314.50,511.00
14,2024-01-01 00:00:00,2024-01-01,00:00:00,30.12,-91.00,3.00,16.20,511.00
17,2024-01-01 00:00:01,2024-01-01,00:00:01,30.06,-90.87,5.70,285.20,511.00


In [96]:
print("SOG summary where Heading = 511\n")

display(
    AIS_01_01.loc[
        AIS_01_01["Heading"] == 511,
        "SOG"
    ]
    .describe()
    .round(2)
)

print("\nMaximum SOG where Heading = 511:")

print(
    AIS_01_01.loc[
        AIS_01_01["Heading"] == 511,
        "SOG"
    ].max()
)

SOG summary where Heading = 511



count   3,827,373.00
mean            1.21
std             5.64
min             0.00
25%             0.00
50%             0.00
75%             0.10
max           102.30
Name: SOG, dtype: float64


Maximum SOG where Heading = 511:
102.3


In [97]:
print("Non-sentinel Heading values (>360 and not 511)\n")

non_sentinel_heading_mask = (
    (AIS_01_01["Heading"] > 360) &
    (AIS_01_01["Heading"] != 511)
)

display(
    AIS_01_01.loc[
        non_sentinel_heading_mask,
        movement_cols
    ]
)

Non-sentinel Heading values (>360 and not 511)



,BaseDateTime,Date,Time,LAT,LON,SOG,COG,Heading
1210888,2024-01-01 04:36:14,2024-01-01,04:36:14,29.94,-93.67,0.10,14.00,510.00
3037215,2024-01-01 11:34:13,2024-01-01,11:34:13,29.29,-89.58,8.00,181.50,510.00
4951397,2024-01-01 18:37:30,2024-01-01,18:37:30,33.88,-118.49,0.30,340.80,444.00


In [98]:
print("Consistency check — SOG 102.3 vs Heading 511\n")

sog_102_mask = AIS_01_01["SOG"] == 102.3
heading_511_mask = AIS_01_01["Heading"] == 511

total_102 = sog_102_mask.sum()

aligned = (
    sog_102_mask &
    heading_511_mask
).sum()

not_aligned = total_102 - aligned

print("Total SOG = 102.3:", total_102)
print("With Heading = 511:", aligned)
print("Without Heading = 511:", not_aligned)

Consistency check — SOG 102.3 vs Heading 511

Total SOG = 102.3: 16571
With Heading = 511: 8412
Without Heading = 511: 8159


In [99]:
print("Sample — SOG 102.3 WITH placeholder pattern (Heading = 511)\n")

display(
    AIS_01_01.loc[
        (AIS_01_01["SOG"] == 102.3) &
        (AIS_01_01["Heading"] == 511),
        movement_cols
    ]
    .sort_values("BaseDateTime")
    .head(20)
)

Sample — SOG 102.3 WITH placeholder pattern (Heading = 511)



,BaseDateTime,Date,Time,LAT,LON,SOG,COG,Heading
1941,2024-01-01 00:00:02,2024-01-01,00:00:02,41.17,-73.18,102.30,360.00,511.00
1096,2024-01-01 00:00:03,2024-01-01,00:00:03,41.72,-87.54,102.30,360.00,511.00
1033,2024-01-01 00:00:04,2024-01-01,00:00:04,29.12,-90.21,102.30,360.00,511.00
1999,2024-01-01 00:00:04,2024-01-01,00:00:04,38.94,-75.32,102.30,360.00,511.00
2457,2024-01-01 00:00:07,2024-01-01,00:00:07,18.01,-66.76,102.30,360.00,511.00
1889,2024-01-01 00:00:08,2024-01-01,00:00:08,28.89,-90.00,102.30,360.00,511.00
6480,2024-01-01 00:00:12,2024-01-01,00:00:12,28.41,-80.62,102.30,360.00,511.00
6141,2024-01-01 00:00:17,2024-01-01,00:00:17,26.06,-80.13,102.30,360.00,511.00
7055,2024-01-01 00:01:11,2024-01-01,00:01:11,41.17,-73.18,102.30,360.00,511.00
8222,2024-01-01 00:01:13,2024-01-01,00:01:13,27.76,-82.63,102.30,360.00,511.00


In [100]:
print("Sample — SOG 102.3 WITHOUT placeholder pattern\n")

display(
    AIS_01_01.loc[
        (AIS_01_01["SOG"] == 102.3) &
        (AIS_01_01["Heading"] != 511),
        movement_cols
    ]
    .sort_values("BaseDateTime")
    .head(20)
)

Sample — SOG 102.3 WITHOUT placeholder pattern



,BaseDateTime,Date,Time,LAT,LON,SOG,COG,Heading
1083,2024-01-01 00:00:04,2024-01-01,00:00:04,47.98,-122.22,102.30,360.00,36.00
1833,2024-01-01 00:00:07,2024-01-01,00:00:07,30.36,-88.51,102.30,360.00,90.00
4713,2024-01-01 00:00:09,2024-01-01,00:00:09,25.78,-80.18,102.30,360.00,149.00
2746,2024-01-01 00:00:11,2024-01-01,00:00:11,29.16,-89.25,102.30,360.00,137.00
10922,2024-01-01 00:01:14,2024-01-01,00:01:14,47.98,-122.22,102.30,360.00,35.00
5315,2024-01-01 00:01:17,2024-01-01,00:01:17,30.36,-88.51,102.30,360.00,90.00
8519,2024-01-01 00:01:19,2024-01-01,00:01:19,25.78,-80.18,102.30,360.00,149.00
7147,2024-01-01 00:01:20,2024-01-01,00:01:20,29.15,-89.25,102.30,360.00,147.00
7317,2024-01-01 00:01:32,2024-01-01,00:01:32,40.67,-74.01,102.30,360.00,19.00
10075,2024-01-01 00:02:09,2024-01-01,00:02:09,45.65,-122.74,102.30,360.00,16.00


In [101]:
print("High SOG values (> 50 knots) — summary\n")

high_sog_mask = (AIS_01_01["SOG"] > 50) & (AIS_01_01["SOG"] < 100)

high_sog_count = high_sog_mask.sum()

print("Number of rows with SOG > 50:", high_sog_count)

print("\nUnique high SOG values:")

display(
    sorted(
        AIS_01_01.loc[
            high_sog_mask,
            "SOG"
        ]
        .dropna()
        .unique()
    )
)

print("\nSample rows with high SOG values:")

display(
    AIS_01_01.loc[
        high_sog_mask,
        movement_cols
    ]
    .sort_values("SOG", ascending=False)
    .head(20)
)

High SOG values (> 50 knots) — summary

Number of rows with SOG > 50: 72

Unique high SOG values:


[np.float64(50.1),
 np.float64(50.3),
 np.float64(50.5),
 np.float64(50.6),
 np.float64(50.8),
 np.float64(51.2),
 np.float64(51.6),
 np.float64(51.7),
 np.float64(51.9),
 np.float64(52.1),
 np.float64(52.3),
 np.float64(52.5),
 np.float64(52.6),
 np.float64(52.7),
 np.float64(52.9),
 np.float64(53.0),
 np.float64(53.3),
 np.float64(53.4),
 np.float64(53.6),
 np.float64(53.8),
 np.float64(53.9),
 np.float64(54.0),
 np.float64(54.1),
 np.float64(54.2),
 np.float64(54.4),
 np.float64(55.0),
 np.float64(55.4),
 np.float64(55.7),
 np.float64(56.2),
 np.float64(57.0),
 np.float64(57.2),
 np.float64(57.5),
 np.float64(57.6),
 np.float64(60.0),
 np.float64(60.2),
 np.float64(61.4),
 np.float64(62.5),
 np.float64(62.9),
 np.float64(64.3),
 np.float64(65.1),
 np.float64(66.9),
 np.float64(67.4),
 np.float64(67.5),
 np.float64(68.6),
 np.float64(72.4),
 np.float64(72.6),
 np.float64(78.0),
 np.float64(78.2),
 np.float64(80.9),
 np.float64(81.3),
 np.float64(83.0),
 np.float64(83.9),
 np.float64(


Sample rows with high SOG values:


,BaseDateTime,Date,Time,LAT,LON,SOG,COG,Heading
1912241,2024-01-01 07:13:10,2024-01-01,07:13:10,31.15,-119.67,97.20,139.20,511.00
2282302,2024-01-01 07:11:40,2024-01-01,07:11:40,31.18,-119.70,94.60,138.50,511.00
1907822,2024-01-01 07:10:09,2024-01-01,07:10:09,31.21,-119.73,92.10,137.70,511.00
1893004,2024-01-01 07:08:42,2024-01-01,07:08:42,31.24,-119.76,89.90,136.80,511.00
5677218,2024-01-01 21:22:29,2024-01-01,21:22:29,24.58,-81.79,87.40,2.00,511.00
1884974,2024-01-01 07:06:40,2024-01-01,07:06:40,31.28,-119.80,86.90,135.70,511.00
1867078,2024-01-01 07:05:39,2024-01-01,07:05:39,31.29,-119.82,85.60,135.10,511.00
1858820,2024-01-01 07:04:11,2024-01-01,07:04:11,31.32,-119.84,83.90,132.80,511.00
6638729,2024-01-01 07:02:40,2024-01-01,07:02:40,31.34,-119.88,83.00,128.20,511.00
5692221,2024-01-01 21:23:42,2024-01-01,21:23:42,24.59,-81.80,81.30,198.60,511.00


### Heading and COG Validation — Summary

Heading = 511 exceeds valid range (0–359). Frequently appears alongside known AIS sentinel values:
- SOG = 102.3
- COG = 360
- Heading = 511

These represent "not available" navigation data per ITU-R M.1371 specification:

| Field | Valid Range | Sentinel Value |
|-------|-------------|----------------|
| Heading | 0–359° | 511 |
| SOG | 0–102.2 kn | 102.3 |
| COG | 0–359.9° | 360 |

Pattern confirmed as expected placeholder behaviour. No data corruption detected.

---

Sentinel values verified against ITU-R M.1371. COG and Heading approved for navigation analysis. Heading = 511 and COG = 360 handled as null-equivalent in bearing and course calculations.

In [102]:
print("Timestamp range\n")

print(
    AIS_01_01["BaseDateTime"].min(),
    "to",
    AIS_01_01["BaseDateTime"].max()
)

Timestamp range

2024-01-01 00:00:00 to 2024-01-01 23:59:59


---

## Profiling Summary

### Validation Results

| Domain | Status | Decision |
|--------|--------|----------|
| Identifier (MMSI) | ✓ Valid | Retain all. Flag MMSI = 0 as metadata gap. |
| Position (LAT/LON) | ✓ Valid | Retain all. No exclusions. |
| Movement (SOG/COG/Heading) | ✓ Valid | Treat sentinel values as NULL. |
| Dimensions (L/W/Draft) | ✓ Valid | Treat zero values as NULL. |
| Temporal | ✓ Valid | Retain all. Dedup zero-time intervals during curation. |

### Data Handling Rules

| Condition | Action |
|-----------|--------|
| MMSI = 0 | Retain. Exclude from MMSI-based joins. |
| Length/Width = 0 | Retain. Treat as NULL for dimension analysis. |
| SOG = 102.3 | Treat as NULL. Exclude from speed calculations. |
| COG = 360 | Treat as NULL. Exclude from course calculations. |
| Heading = 511 | Treat as NULL. Exclude from bearing calculations. |
| Duplicate MMSI + BaseDateTime | Available for deduplication during curation. |

### Overall Assessment

**Finding:** No high-risk integrity issues identified. Observed anomalies consistent with expected AIS transmission behaviour and missing metadata patterns.

**Decision:** Dataset approved for data curation.

**Action:** Proceed to Step 2 — Data Cleaning and Curation.

**Impact:** Full dataset (7.3M rows) available for downstream processing with documented handling rules.

---

## Sources

- **NOAA Marine Cadastre** — AIS Data Dictionary  
  https://coast.noaa.gov/data/marinecadastre/ais/data-dictionary.pdf

- **U.S. Coast Guard Navigation Center** — AIS Guidance  
  https://www.navcen.uscg.gov/ais

- **ITU-R M.1371** — Technical characteristics for AIS  
  Sentinel values: Heading = 511, SOG = 102.3, COG = 360

- **Maritime Classification References**  
  IMO, ABS, Lloyd's Register — vessel proportion guidance

---

## Cross-Day Stability Check

Quick validation that anomaly patterns observed on Day 1 remain consistent across sampled days before closing Step 1.

In [103]:
sample_days = [1, 10, 20, 30]
stability_cols = ["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "Heading"]

sample_files = []
for day in sample_days:
    file_name = f"AIS_2024_01_{day:02d}.csv"
    file_path = RAW_DIR / file_name
    if file_path.exists():
        sample_files.append((file_name, file_path))
        print(f"Found: {file_name}")
    else:
        print(f"Missing: {file_name}")

print(f"\nFiles available for stability check: {len(sample_files)}")

Found: AIS_2024_01_01.csv
Found: AIS_2024_01_10.csv
Found: AIS_2024_01_20.csv
Found: AIS_2024_01_30.csv

Files available for stability check: 4


In [104]:
stability_results = []

for file_name, file_path in sample_files:
    print(f"Processing {file_name}...")
    df = pd.read_csv(file_path, usecols=stability_cols)
    
    row_count = len(df)
    dup_count = df.duplicated(subset=["MMSI", "BaseDateTime"], keep=False).sum()
    null_mmsi = df["MMSI"].isna().sum()
    null_lat = df["LAT"].isna().sum()
    null_lon = df["LON"].isna().sum()
    heading_511 = (df["Heading"] == 511).sum()
    sog_102_3 = (df["SOG"] == 102.3).sum()
    invalid_lat = ((df["LAT"] < -90) | (df["LAT"] > 90)).sum()
    invalid_lon = ((df["LON"] < -180) | (df["LON"] > 180)).sum()
    high_sog = ((df["SOG"] > 50) & (df["SOG"] < 100)).sum()
    
    stability_results.append({
        "file_name": file_name,
        "row_count": row_count,
        "duplicate_mmsi_timestamp_count": dup_count,
        "null_mmsi_count": null_mmsi,
        "null_lat_count": null_lat,
        "null_lon_count": null_lon,
        "heading_511_count": heading_511,
        "sog_102_3_count": sog_102_3,
        "invalid_lat_count": invalid_lat,
        "invalid_lon_count": invalid_lon,
        "high_sog_over_50_count": high_sog
    })

print("\nDone.")

Processing AIS_2024_01_01.csv...
Processing AIS_2024_01_10.csv...
Processing AIS_2024_01_20.csv...
Processing AIS_2024_01_30.csv...

Done.


In [105]:
stability_df = pd.DataFrame(stability_results)

print("Cross-Day Stability Summary\n")
display(stability_df)

Cross-Day Stability Summary



,file_name,row_count,duplicate_mmsi_timestamp_count,null_mmsi_count,null_lat_count,null_lon_count,heading_511_count,sog_102_3_count,invalid_lat_count,invalid_lon_count,high_sog_over_50_count
0,AIS_2024_01_01.csv,7296275,516,0,0,0,3827373,16571,0,0,72
1,AIS_2024_01_10.csv,7024515,298,0,0,0,3583119,12373,0,0,76
2,AIS_2024_01_20.csv,7420105,376,0,0,0,3741521,16430,0,0,25
3,AIS_2024_01_30.csv,7644363,646,0,0,0,3909184,18460,0,0,52


### Stability Check — Conclusion

Sampled days checked: Day 1, Day 10, Day 20, Day 30.

No new anomaly classes observed. Sentinel values (Heading 511, SOG 102.3) appear consistently across days. Duplicate patterns and null counts remain proportional to row volume.

Validation patterns from Day 1 remain broadly consistent. The profiling framework is applicable to the full January 2024 dataset.

**Step 1 — Data Loading and Profiling: Complete.**